# GAPS 투자계획서 초안

## 전략명
**Adaptive Theme Rotation with Target Vol Overlay**  
고변동성 상승장(Spot Up, Vol Up)에서 테마 모멘텀은 추종하되, VKOSPI 기반 위험노출 조절로 과열 리스크를 관리하는 전략.

## 한 줄 투자철학
강한 테마는 따라가되, 변동성이 과도하게 상승한 구간에서는 수익률보다 생존 가능성을 우선한다.

## 작성 목적
- 2026년 6월~8월 GAPS 운용기간의 투자계획서 초안 작성
- `universe.xlsx` 내 국내 상장 ETF만 사용
- 자산군/세부자산/개별 ETF 비중 제한을 준수하는 초기 포트폴리오 구성
- 향후 ETF 구성종목, 비중, 지수 방법론 조사 결과를 반영할 수 있는 선정 프레임 구축

## 1. 시장 인식: 삼성증권 2026년 5월 리포트 기반

전략의 출발점은 삼성증권 Derivatives Weekly **「Spot Up, Vol Up: Target Vol 전략의 속도조절 기능」(2026.05.08)** 이다. 이 리포트는 2026년 5월 한국 시장을 단순 강세장이 아니라, 주가지수와 변동성이 동시에 상승하는 **Spot Up, Vol Up** 국면으로 해석한다.

리포트의 핵심을 투자계획서 논리로 바꾸면 다음과 같다.

| 리포트에서 관찰한 내용 | 투자계획서에 반영한 의사결정 |
|---|---|
| KOSPI200이 1,000pt를 돌파하는 과정에서 VKOSPI도 60%를 상회 | 강세장은 인정하되, 위험자산을 70% 상한까지 즉시 채우지 않고 초기 65%로 시작 |
| Spot Up & VIX Up은 미국 기준으로도 자주 나타나는 전형적 패턴이 아니라 이례적 국면이라고 설명 | 단순 모멘텀 추격이 아니라 변동성 기반 속도조절 장치를 전략 중심에 배치 |
| 주가 상승과 변동성 상승이 동행하면 타깃볼 투자자는 포트폴리오 변동성을 낮추기 위해 델타를 축소할 수 있음 | VKOSPI가 높은 구간에서는 CD/MMF/단기채 비중을 유지하고, 위험자산 추가 매수는 변동성 피크아웃 이후로 미룸 |
| 5월 외국인 선물매도는 하락 베팅이라기보다 변동성 상승에 따른 타깃볼성 델타 축소로 해석 | 단기 조정 위험을 인정하되, 변동성이 진정되면 헤지 청산이 오히려 주가 하방을 지지할 수 있다고 판단 |
| KOSDAQ150은 2차전지와 반도체 소부장 모멘텀이 필요하고 ETF 수급은 관망이라고 언급 | KOSDAQ150은 코어가 아니라 3% 수준의 위성 포지션으로 제한 |

따라서 본 전략은 **공격적 테마 추종 + 변동성 기반 위험노출 조절**로 설계한다. 핵심은 상승장을 포기하지 않는 것이지만, VKOSPI가 높은 상태에서 테마 ETF를 무리하게 늘리지 않는 것이다.

초기 운용 원칙은 다음과 같다.

1. 위험자산 비중은 65%로 시작한다. 규정상 70%까지 가능하지만, 삼성증권 리포트가 지적한 Spot Up, Vol Up 리스크를 반영해 5%p의 완충을 둔다.
2. VKOSPI가 60 아래로 안정되고 KOSPI200 추세가 유지되면 위험자산을 70%까지 확대할 수 있다.
3. VKOSPI가 70을 넘거나 KOSPI200이 단기 추세선을 이탈하면 위험자산을 55% 이하로 낮춘다.
4. 테마 ETF는 universe에 명시된 Theme를 기준으로 묶고, 같은 테마군에서는 대표 ETF 1개만 편입한다.
5. 대표 ETF 선정 시 운용보수, AUM, 최근 거래대금, 가격 모멘텀, 변동성을 함께 본다.


In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

# 데이터 경로: 노트북을 현재 폴더에서 실행하는 경우를 우선하고, 기존 하드코딩 경로는 후순위 fallback으로 둔다.
DATA_DIR_CANDIDATES = [
    Path.cwd(),
    Path.cwd().parent / 'input',
    Path.cwd() / 'input',
]
DATA_DIR = next((p for p in DATA_DIR_CANDIDATES if (p / 'universe.xlsx').exists()), DATA_DIR_CANDIDATES[0])
UNIVERSE_PATH = DATA_DIR / 'universe.xlsx'
FACTSHEET_DIR = DATA_DIR / 'factsheet'
PRICE_PATH = DATA_DIR / 'DBGAPS_2021_2026_수정주가_일간비영업일제외.xlsx'
VOLUME_PATH = DATA_DIR / 'DBGAPS_2021_2026_거래량_일간비영업일제외.xlsx'
VALUE_PATH = DATA_DIR / 'DBGAPS_2021_2026_거래대금_일간비영업일제외.xlsx'
VKOSPI_PATH = DATA_DIR / 'VKOSPI.xlsx'
MARKET_INDEX_PATH = DATA_DIR / '코스피, 코스피200, 코스닥150.xlsx'
REPORT_PATH = DATA_DIR / '자료' / '시장전망' / '삼성증권_DerivativesWeekly_SpotUpVolUp_TargetVol_20260508.pdf'

# ===== 백테스트 핵심 파라미터 =====
TRADING_DAYS = 252
BACKTEST_START_REQUESTED = pd.Timestamp('2025-06-02')
BACKTEST_END_REQUESTED = pd.Timestamp('2026-05-22')

# 거래비용: 대회 공식 기준은 국내 ETF 거래수수료 0.1%, 제세금 없음이다.
# 슬리피지는 공식 NAV 산정 비용이 아니므로 비용 반영 후 표에는 별도 차감하지 않는다.
COMMISSION_RATE = 0.00100          # 대회 기준 국내 ETF 거래수수료 0.1%
TAX_RATE = 0.0                     # 제세금 없음
APPLY_SLIPPAGE_IN_OFFICIAL_BACKTEST = False
SLIPPAGE_LARGE_ETF = 0.0           # 공식 NAV 비용에는 슬리피지 미반영
SLIPPAGE_THEME_SMALL_ETF = 0.0     # 공식 NAV 비용에는 슬리피지 미반영
INCLUDE_INITIAL_TRADING_COST = True # 최초 포트폴리오 설정 매매비용 포함
LARGE_LIQUID_TICKERS = {
    'A069500',  # KODEX 200
    'A102110',  # TIGER 200
    'A229200',  # KODEX 코스닥150
    'A360750',  # TIGER 미국S&P500
    'A360200',  # ACE 미국S&P500
    'A379800',  # KODEX 미국S&P500
    'A133690',  # TIGER 미국나스닥100
    'A411060',  # ACE KRX금현물
    'A459580',  # KODEX CD금리액티브(합성)
    'A423160',  # KODEX KOFR금리액티브(합성)
    'A488770',  # KODEX 머니마켓액티브
    'A157450',  # TIGER 단기통안채
    'A0061Z0',  # RISE 단기특수은행채액티브
    'A453850',  # ACE 미국30년국채액티브(H)
}

# 무위험수익률/Rf 및 Sortino MAR: None이면 백테스트 기간의 CD금리 ETF 평균 일수익률을 연환산해 사용한다.
RF_PROXY_TICKER = 'A459580'
RF_PROXY_DESCRIPTION = 'KODEX CD금리액티브(합성) 일별 수익률 평균의 252영업일 연환산값'
RISK_FREE_RATE_ANNUAL = None
MAR_ANNUAL = None

# 룩어헤드 방지 및 리밸런싱 체결 가정
SIGNAL_LOOKBACK_WINDOWS = (20, 60)
SIGNAL_EXECUTION_ASSUMPTION = 't일 종가 신호(VKOSPI/KOSPI200) 확인 후 t+1영업일 시가 체결'
REBALANCE_EXECUTION_ASSUMPTION = '월말 종가로 신호 확정, 다음 영업일(월초) 시가에 리밸런싱'
PRICE_EXECUTION_PROXY = '현재 파일에는 일별 수정주가 종가만 있어 t+1 수정종가를 월초 시가 체결의 보수적 프록시로 사용'

# 계획서 규칙형 백테스트 파라미터
# Target Vol 효과를 '방어 overlay'로 분리하려면 65% 초과 위험확대를 끄고,
# 70% 확대 옵션은 별도 민감도 분석으로 켠다.
PLAN_RULES_ALLOW_RISK_EXPANSION = False
PLAN_RULES_REQUIRE_FALLING_VKOSPI_FOR_EXPANSION = True
PLAN_RULES_BASE_RISK_BUDGET = 0.65
PLAN_RULES_EXPANSION_RISK_BUDGET = 0.70
PLAN_RULES_DEFENSE_RISK_BUDGET = 0.55
PLAN_RULES_FORCED_DEFENSE_RISK_BUDGET = 0.50
PLAN_RULES_REBALANCE_BAND_ABS = 0.01
PLAN_RULES_RISK_SOFT_BAND = 0.74
PLAN_RULES_RISK_HARD_BAND = 0.76
PLAN_RULES_DOMESTIC_SECTOR_SOFT_BAND = 0.17
PLAN_RULES_DOMESTIC_SECTOR_HARD_BAND = 0.18
PLAN_RULES_SINGLE_ETF_SOFT_BAND = 0.23
PLAN_RULES_SINGLE_ETF_HARD_BAND = 0.25
PLAN_RULES_SINGLE_ETF_TRIM_TARGET = 0.205


universe = pd.read_excel(UNIVERSE_PATH, sheet_name='기본정보')

backtest_parameters = pd.DataFrame([
    {'Parameter': 'Backtest request', 'Value': f'{BACKTEST_START_REQUESTED.date()} ~ {BACKTEST_END_REQUESTED.date()}'},
    {'Parameter': 'Contest ETF commission', 'Value': f'{COMMISSION_RATE:.2%}'},
    {'Parameter': 'Tax', 'Value': f'{TAX_RATE:.2%}'},
    {'Parameter': 'Slippage in official NAV', 'Value': 'Not deducted' if not APPLY_SLIPPAGE_IN_OFFICIAL_BACKTEST else 'Deducted'},
    {'Parameter': 'Initial trading cost', 'Value': 'Included' if INCLUDE_INITIAL_TRADING_COST else 'Excluded'},
    {'Parameter': 'Rf source', 'Value': RF_PROXY_DESCRIPTION if RISK_FREE_RATE_ANNUAL is None else f'User fixed {RISK_FREE_RATE_ANNUAL:.2%}'},
    {'Parameter': 'MAR for Sortino', 'Value': 'Same as Rf' if MAR_ANNUAL is None else f'{MAR_ANNUAL:.2%}'},
    {'Parameter': 'Signal execution', 'Value': SIGNAL_EXECUTION_ASSUMPTION},
    {'Parameter': 'Monthly rebalance', 'Value': REBALANCE_EXECUTION_ASSUMPTION},
    {'Parameter': 'Plan-rules risk expansion', 'Value': 'ON, only under strict falling-VKOSPI confirmation' if PLAN_RULES_ALLOW_RISK_EXPANSION else 'OFF for Target Vol effect table; 70% expansion is separate sensitivity'},
    {'Parameter': 'Plan-rules monthly trade band', 'Value': f'{PLAN_RULES_REBALANCE_BAND_ABS:.1%} absolute weight drift for scheduled rotation trades'},
    {'Parameter': 'Plan-rules risk band', 'Value': f'{PLAN_RULES_RISK_SOFT_BAND:.0%} monthly trim / {PLAN_RULES_RISK_HARD_BAND:.0%} mid-month trim'},
    {'Parameter': 'Plan-rules sector/single ETF band', 'Value': f'sector {PLAN_RULES_DOMESTIC_SECTOR_SOFT_BAND:.0%}/{PLAN_RULES_DOMESTIC_SECTOR_HARD_BAND:.0%}, single {PLAN_RULES_SINGLE_ETF_SOFT_BAND:.0%}/{PLAN_RULES_SINGLE_ETF_HARD_BAND:.0%}'},
])
display(backtest_parameters)
universe.head()



,Parameter,Value
0,Backtest request,2025-06-02 ~ 2026-05-22
1,Contest ETF commission,0.10%
2,Tax,0.00%
3,Slippage in official NAV,Not deducted
4,Initial trading cost,Included
5,Rf source,KODEX CD금리액티브(합성) 일별 수익률 평균의 252영업일 연환산값
6,MAR for Sortino,Same as Rf
7,Signal execution,t일 종가 신호(VKOSPI/KOSPI200) 확인 후 t+1영업일 시가 체결
8,Monthly rebalance,"월말 종가로 신호 확정, 다음 영업일(월초) 시가에 리밸런싱"
9,Plan-rules risk expansion,OFF for Target Vol effect table; 70% expansion...


,Ticker,Name,Exchange,AUM,Class1,Class2,Target Market,Region,Asset Type,Management Type,Hedged,Total Return,Sector,Theme,Style,Target Ticker,Target Exchange,Dividend Rate,Expense Ratio
0,A411060,ACE KRX금현물,KRX,52302.951396,위험,FX 및 원자재,Gold,Global,Spot,Passive,False,False,Commodity,Gold,Market Cap / Index,GOLD,TVC,0.000032,0.0031
1,A144600,KODEX 은선물(H),KRX,14590.155025,위험,FX 및 원자재,Silver,Global,Futures,Passive,True,False,Commodity,Silver,Market Cap / Index,SI1!,COMEX,0.002862,0.0073
2,A132030,KODEX 골드선물(H),KRX,4821.274136,위험,FX 및 원자재,Gold,Global,Futures,Passive,True,False,Commodity,Gold,Market Cap / Index,GC1!,COMEX,0.004881,0.0073
3,A261220,KODEX WTI원유선물(H),KRX,1428.223927,위험,FX 및 원자재,Crude Oil,Global,Futures,Passive,True,False,Commodity,Crude Oil,Market Cap / Index,CL1!,NYMEX,NaN,0.0040
4,A261240,KODEX 미국달러선물,KRX,723.625729,위험,FX 및 원자재,US Dollar,United States,Futures,Passive,False,False,Foreign Exchange,US Dollar,Market Cap / Index,261240,KRX,0.008864,0.0028


In [2]:
overview = {
    '전체 ETF 수': len(universe),
    '위험자산 수': int((universe['Class1'] == '위험').sum()),
    '안전자산 수': int((universe['Class1'] == '안전').sum()),
}
overview

{'전체 ETF 수': 188, '위험자산 수': 138, '안전자산 수': 50}

In [3]:
class2_summary = (
    universe.groupby(['Class1', 'Class2'])
    .agg(ETF수=('Ticker', 'count'), AUM합계=('AUM', 'sum'), 평균보수=('Expense Ratio', 'mean'))
    .sort_values(['Class1', 'AUM합계'], ascending=[True, False])
)
class2_summary

ETF수          AUM합계      평균보수
Class1 Class2                                    
안전     금리연계형/초단기채권    11  244019.496518  0.000927
       국내채권_회사채        9   93999.296014  0.001267
       해외채권_종합        11   46528.688820  0.002336
       국내채권_종합        12   33268.929342  0.001217
       해외채권_회사채        7    5359.292293  0.002800
위험     해외주식_지수        47  539775.596167  0.002915
       국내주식_지수        17  521029.184448  0.001094
       국내주식_섹터        36  304058.578168  0.003852
       해외주식_섹터        31  180110.453984  0.002739
       FX 및 원자재        7   74659.973410  0.004757

## 2. 운용 제약 조건

이미 주어진 세부자산별 상한은 전략 설계의 가장 중요한 안전장치다. 다만 같은 세부자산 안에서도 유사 ETF가 많기 때문에, 계획서에서는 **상한 준수 + 대표 ETF 선별**을 같이 제시한다.

| 구분 | 상한 |
|---|---:|
| 위험자산 전체 | 70% |
| 안전자산 전체 | 100% |
| 개별 ETF | 20% |
| 국내주식_지수 | 30% |
| 국내주식_섹터 | 15% |
| 해외주식_지수 | 30% |
| 해외주식_섹터 | 10% |
| FX 및 원자재 | 20% |
| 국내채권_종합 | 50% |
| 국내채권_회사채 | 30% |
| 해외채권_종합 | 50% |
| 해외채권_회사채 | 30% |
| 금리연계형/초단기채권 | 50% |

In [4]:
CONSTRAINTS = {
    'Class1': {
        '위험': 0.70,
        '안전': 1.00,
    },
    'Class2': {
        '국내주식_지수': 0.30,
        '국내주식_섹터': 0.15,
        '해외주식_지수': 0.30,
        '해외주식_섹터': 0.10,
        'FX 및 원자재': 0.20,
        '국내채권_종합': 0.50,
        '국내채권_회사채': 0.30,
        '해외채권_종합': 0.50,
        '해외채권_회사채': 0.30,
        '금리연계형/초단기채권': 0.50,
    },
    'single_etf': 0.20,
}

def validate_portfolio(portfolio, universe=universe, constraints=CONSTRAINTS):
    """portfolio: DataFrame with Ticker, Weight columns."""
    pf = portfolio.merge(universe, on='Ticker', how='left')
    checks = []

    for cls, limit in constraints['Class1'].items():
        weight = pf.loc[pf['Class1'] == cls, 'Weight'].sum()
        checks.append({'구분': f'Class1={cls}', '비중': weight, '상한': limit, '통과': weight <= limit + 1e-9})

    for cls, limit in constraints['Class2'].items():
        weight = pf.loc[pf['Class2'] == cls, 'Weight'].sum()
        checks.append({'구분': f'Class2={cls}', '비중': weight, '상한': limit, '통과': weight <= limit + 1e-9})

    max_single = pf['Weight'].max()
    checks.append({'구분': '개별 ETF 최대', '비중': max_single, '상한': constraints['single_etf'], '통과': max_single <= constraints['single_etf'] + 1e-9})
    checks.append({'구분': '총합', '비중': pf['Weight'].sum(), '상한': 1.00, '통과': abs(pf['Weight'].sum() - 1.00) < 1e-9})

    return pd.DataFrame(checks)

## 3. 대표 ETF 선정 기준

비슷한 ETF가 여러 개 있을 때는 무조건 많이 담지 않고, 같은 역할을 하는 ETF 중 하나만 고른다. 이때 기준은 다음 순서로 적용한다.

1. **전략 적합성**: 해당 ETF가 원하는 테마에 가장 직접적으로 노출되는가?
2. **운용보수**: 동일 노출이면 총보수가 낮은 ETF를 우선한다.
3. **유동성/AUM**: 보수가 낮아도 AUM이 지나치게 작으면 제외한다.
4. **지수 방법론**: 구성종목 집중도, 리밸런싱 방식, 편입 조건이 전략 논리와 맞는가?
5. **구성종목 중복도**: 이미 보유한 ETF와 상위 구성종목이 과도하게 겹치면 제외한다.

현재 노트북에서는 `universe.xlsx`로 확인 가능한 운용보수와 AUM만 먼저 반영한다. 지수 방법론과 구성종목 중복도는 후속 조사 후 확정한다.

In [5]:
def load_dbgaps_xlsx(path):
    # DBGAPS 엑셀: 9행 코드, 10행 코드명, 15행부터 일별 데이터.
    raw = pd.read_excel(path, sheet_name='Sheet1', header=None)
    codes = raw.iloc[8, 1:].dropna().astype(str).str.strip().tolist()
    names = raw.iloc[9, 1:1 + len(codes)].tolist()
    data = raw.iloc[14:, :1 + len(codes)].copy()
    data.columns = ['Date'] + codes
    data['Date'] = pd.to_datetime(data['Date'])
    data = data.set_index('Date').apply(pd.to_numeric, errors='coerce')
    return data, dict(zip(codes, names))

price, code_names = load_dbgaps_xlsx(PRICE_PATH)
volume, _ = load_dbgaps_xlsx(VOLUME_PATH)
trading_value, _ = load_dbgaps_xlsx(VALUE_PATH)
returns = price.pct_change(fill_method=None)
latest_date = price.dropna(how='all').index.max()

print('latest_date:', latest_date.date())
print('price:', price.shape, 'volume:', volume.shape, 'trading_value:', trading_value.shape)


def total_return(series, days):
    s = series.dropna()
    if len(s) <= days:
        return np.nan
    return s.iloc[-1] / s.iloc[-days - 1] - 1


def build_metrics(universe, price, volume, trading_value):
    rows = []
    for ticker in universe['Ticker'].astype(str):
        if ticker not in price.columns:
            continue
        p = price[ticker].dropna()
        if len(p) < 30:
            continue
        r = returns[ticker].dropna()
        rows.append({
            'Ticker': ticker,
            'LatestDate': p.index.max(),
            'Obs': len(p),
            'Ret_1M': total_return(p, 21),
            'Ret_3M': total_return(p, 63),
            'Ret_6M': total_return(p, 126),
            'Ret_12M': total_return(p, 252),
            'Vol_20D': r.iloc[-20:].std() * np.sqrt(252) if len(r) >= 20 else np.nan,
            'Vol_60D': r.iloc[-60:].std() * np.sqrt(252) if len(r) >= 60 else np.nan,
            'Volume_20D': volume[ticker].dropna().iloc[-20:].mean() if ticker in volume.columns else np.nan,
            'TradingValue_20D': trading_value[ticker].dropna().iloc[-20:].mean() if ticker in trading_value.columns else np.nan,
        })
    return pd.DataFrame(rows)

metrics = build_metrics(universe, price, volume, trading_value)
scored = universe.merge(metrics, on='Ticker', how='left')


def pct_rank(s, ascending=True):
    return s.rank(pct=True, ascending=ascending)

valid = scored[scored['Ret_3M'].notna()].copy()
valid['Ret_1M_rank'] = pct_rank(valid['Ret_1M'])
valid['Ret_3M_rank'] = pct_rank(valid['Ret_3M'])
valid['Ret_6M_rank'] = pct_rank(valid['Ret_6M'])
valid['AUM_rank'] = pct_rank(valid['AUM'])
valid['Liquidity_rank'] = pct_rank(valid['TradingValue_20D'])
valid['Fee_rank'] = pct_rank(-valid['Expense Ratio'].fillna(valid['Expense Ratio'].median()))
valid['LowVol_rank'] = pct_rank(-valid['Vol_20D'].fillna(valid['Vol_20D'].median()))

# 테마 ETF 점수: 모멘텀을 보되, 고변동성 국면이므로 유동성/보수/변동성도 같이 반영
valid['ThemeScore'] = (
    0.22 * valid['Ret_1M_rank']
    + 0.25 * valid['Ret_3M_rank']
    + 0.18 * valid['Ret_6M_rank']
    + 0.15 * valid['Liquidity_rank']
    + 0.10 * valid['AUM_rank']
    + 0.07 * valid['Fee_rank']
    + 0.03 * valid['LowVol_rank']
)

# 안전자산 점수: 낮은 변동성, 유동성, 낮은 보수를 더 크게 반영
valid['SafeScore'] = (
    0.30 * valid['LowVol_rank']
    + 0.25 * valid['Liquidity_rank']
    + 0.20 * valid['AUM_rank']
    + 0.15 * valid['Fee_rank']
    + 0.10 * valid['Ret_3M_rank']
)

cols = ['Ticker', 'Name', 'Class1', 'Class2', 'Theme', 'AUM', 'Expense Ratio', 'Ret_1M', 'Ret_3M', 'Ret_6M', 'Vol_20D', 'TradingValue_20D', 'ThemeScore']
valid[cols].sort_values('ThemeScore', ascending=False).head(10)


latest_date: 2026-05-22
price: (1321, 188) volume: (1321, 188) trading_value: (1321, 188)


,Ticker,Name,Class1,Class2,Theme,AUM,Expense Ratio,Ret_1M,Ret_3M,Ret_6M,Vol_20D,TradingValue_20D,ThemeScore
87,A102110,TIGER 200,위험,국내주식_지수,Broad Market,67897.056777,0.0005,0.277688,0.470171,1.200160,0.553541,6.472338e+11,0.908112
90,A148020,RISE 200,위험,국내주식_지수,Broad Market,29006.766411,0.0002,0.277737,0.470981,1.200354,0.565418,1.932480e+11,0.902660
85,A069500,KODEX 200,위험,국내주식_지수,Broad Market,164802.595561,0.0015,0.278344,0.468155,1.194801,0.554793,2.495761e+12,0.891489
109,A487240,KODEX AI전력핵심설비,위험,국내주식_섹터,AI,15913.745035,0.0039,0.426574,0.818872,1.257972,0.841278,6.190343e+11,0.888910
102,A091160,KODEX 반도체,위험,국내주식_섹터,Semiconductor,36018.939848,0.0045,0.345633,0.699435,1.774358,0.645329,8.375048e+11,0.888245
110,A395270,HANARO Fn K-반도체,위험,국내주식_섹터,Semiconductor,14663.997501,0.0045,0.430171,0.988925,2.372618,0.617679,3.256171e+11,0.887394
94,A152100,PLUS 200,위험,국내주식_지수,Broad Market,13051.441163,0.0004,0.278853,0.470326,1.198840,0.550605,6.255007e+10,0.879761
105,A395160,KODEX AI반도체,위험,국내주식_섹터,AI; Semiconductor,20947.404372,0.0045,0.378623,0.702111,1.634650,0.612449,2.259665e+11,0.878191
118,A139260,TIGER 200 IT,위험,국내주식_섹터,Broad Market,8198.715885,0.0040,0.456914,0.953243,2.030451,0.650482,2.108546e+11,0.877580
88,A278530,KODEX 200TR,위험,국내주식_지수,Broad Market,52559.358714,0.0005,0.276529,0.468050,1.197732,0.558859,7.737856e+10,0.877314


## 4. 테마별 대표 ETF 선정

`universe.xlsx`의 `Theme` 컬럼을 기준으로 테마를 묶었다. 다만 같은 테마 안에서도 ETF 성격이 겹치는 경우가 많기 때문에, 단순히 점수 1등을 기계적으로 담지 않고 다음 기준으로 대표 ETF를 정한다.

1. **테마 직접성**: 투자 아이디어와 가장 직접적으로 연결되는 ETF인가?
2. **운용보수**: 비슷한 노출이면 낮은 보수를 우선한다.
3. **유동성**: 최근 20영업일 평균 거래대금과 AUM이 충분한가?
4. **성과 지속성**: 1개월만 반짝 오른 ETF보다 3개월/6개월 모멘텀이 같이 살아있는가?
5. **변동성**: VKOSPI 고변동성 국면이므로 동일 기대수익이면 낮은 변동성을 우선한다.
6. **구성종목 중복**: 최종 확정 전 ETF 구성종목과 상위 비중을 확인해 이미 편입한 ETF와 과도하게 겹치면 제외한다.

중요한 점은 세부자산 상한이 이미 존재한다는 것이다. 예를 들어 국내주식_섹터는 최대 15%, 해외주식_섹터는 최대 10%다. 따라서 테마 중복이 포트폴리오 전체를 망가뜨릴 가능성은 제한되어 있지만, 같은 테마 ETF를 여러 개 담으면 전략 설명력이 약해진다. 그래서 **비슷한 ETF는 대표 1개만 선정**한다.


In [6]:
selected_theme_etfs = pd.DataFrame([
    {'Bucket': '국내 AI/반도체', 'Ticker': 'A395160', 'Use': '초기 편입', 'Reason': 'AI 반도체 테마 직접성이 높고 1M/3M/6M 모멘텀이 모두 강함. HANARO Fn K-반도체는 데이터 점수상 대체 후보.'},
    {'Bucket': '국내 AI 전력/인프라', 'Ticker': 'A487240', 'Use': '초기 편입', 'Reason': 'AI 데이터센터 전력 수요라는 별도 수혜 경로. 최근 모멘텀과 거래대금이 강하지만 변동성이 높아 비중 제한.'},
    {'Bucket': '국내 조선', 'Ticker': 'A466920', 'Use': '초기 편입', 'Reason': 'SOL 조선TOP3플러스가 TIGER 조선TOP10보다 AUM/거래대금이 크고 대표성이 높음. 집중도는 추후 구성종목 확인 필요.'},
    {'Bucket': '국내 방산', 'Ticker': 'A449450', 'Use': '초기 편입', 'Reason': 'PLUS K방산은 TIGER K방산&우주보다 AUM과 거래대금이 크고 변동성이 낮아 고변동성 국면에 더 적합.'},
    {'Bucket': '국내 배당/방어', 'Ticker': 'A161510', 'Use': '초기 편입', 'Reason': 'PLUS 고배당주는 국내 방어형 위험자산 역할. 은행고배당보다 테마가 넓고 AUM이 큼.'},
    {'Bucket': '국내 2차전지', 'Ticker': 'A305720', 'Use': '관찰 후보', 'Reason': '2차전지는 반등 후보지만 최근 1개월 모멘텀이 약함. KODEX 2차전지산업은 AUM/거래대금 기준 대표 후보.'},
    {'Bucket': '국내 원자력', 'Ticker': 'A434730', 'Use': '관찰 후보', 'Reason': 'AI 전력 테마의 확장 후보. 단일 ETF이고 변동성이 높아 초기 핵심 편입보다는 교체 후보로 관리.'},
    {'Bucket': '국내 로봇', 'Ticker': 'A445290', 'Use': '관찰 후보', 'Reason': '모멘텀은 있으나 액티브 ETF이고 보수가 높아 후순위.'},
    {'Bucket': '해외 AI 인프라', 'Ticker': 'A487230', 'Use': '초기 편입', 'Reason': '미국 AI 전력 인프라 노출. 국내 AI 전력과 지역이 달라 보완 가능하나 해외섹터 10% 상한 내에서 제한.'},
    {'Bucket': '해외 반도체', 'Ticker': 'A390390', 'Use': '대체 후보', 'Reason': 'KODEX 미국반도체는 보수가 낮고 3M 모멘텀이 강함. 다만 국내 AI/반도체와 중복되므로 초기에는 제외.'},
    {'Bucket': '해외 배당', 'Ticker': 'A458730', 'Use': '초기 편입', 'Reason': 'TIGER 미국배당다우존스는 AUM/거래대금/보수 모두 우수. 해외 위험자산 내 방어형 역할.'},
    {'Bucket': '중국 EV', 'Ticker': 'A371460', 'Use': '관찰 후보', 'Reason': '중국 전기차 반등 후보이나 최근 1개월 흐름이 약해 초기 편입 제외.'},
    {'Bucket': '금', 'Ticker': 'A411060', 'Use': '초기 편입', 'Reason': 'ACE KRX금현물은 선물형보다 구조가 직관적이고 AUM/거래대금이 큼. 주식 변동성 헤지 역할.'},
    {'Bucket': '달러', 'Ticker': 'A261240', 'Use': '대체 후보', 'Reason': '달러 강세 방어용 후보. 다만 AUM이 작아 초기에는 달러SOFR/미국채 후보와 비교 후 결정.'},
    {'Bucket': '엔', 'Ticker': 'A292560', 'Use': '대체 후보', 'Reason': '엔화 반등/위험회피 후보. AUM과 거래대금이 작아 초기 핵심 편입은 보류.'},
    {'Bucket': '원유', 'Ticker': 'A261220', 'Use': '전술 후보', 'Reason': '최근 모멘텀은 강하지만 변동성이 높고 선물형 상품이라 장기 보유보다 짧은 전술 포지션에 적합.'},
])

selected_theme_etfs = selected_theme_etfs.merge(
    valid[['Ticker', 'Name', 'Class2', 'Theme', 'AUM', 'Expense Ratio', 'Ret_1M', 'Ret_3M', 'Ret_6M', 'Vol_20D', 'TradingValue_20D', 'ThemeScore']],
    on='Ticker',
    how='left'
)

selected_theme_etfs[['Bucket', 'Ticker', 'Name', 'Use', 'Class2', 'Theme', 'Expense Ratio', 'Ret_1M', 'Ret_3M', 'Ret_6M', 'Vol_20D', 'TradingValue_20D', 'Reason']]


,Bucket,Ticker,Name,Use,Class2,Theme,Expense Ratio,Ret_1M,Ret_3M,Ret_6M,Vol_20D,TradingValue_20D,Reason
0,국내 AI/반도체,A395160,KODEX AI반도체,초기 편입,국내주식_섹터,AI; Semiconductor,0.004500,0.378623,0.702111,1.634650,0.612449,2.259665e+11,AI 반도체 테마 직접성이 높고 1M/3M/6M 모멘텀이 모두 강함. HANARO ...
1,국내 AI 전력/인프라,A487240,KODEX AI전력핵심설비,초기 편입,국내주식_섹터,AI,0.003900,0.426574,0.818872,1.257972,0.841278,6.190343e+11,AI 데이터센터 전력 수요라는 별도 수혜 경로. 최근 모멘텀과 거래대금이 강하지만 ...
2,국내 조선,A466920,SOL 조선TOP3플러스,초기 편입,국내주식_섹터,Shipbuilding,0.004500,0.015174,0.033279,0.074971,0.434062,1.829848e+11,SOL 조선TOP3플러스가 TIGER 조선TOP10보다 AUM/거래대금이 크고 대표...
3,국내 방산,A449450,PLUS K방산,초기 편입,국내주식_섹터,Defense,0.004500,-0.076227,0.080272,0.349449,0.407595,8.050382e+10,PLUS K방산은 TIGER K방산&우주보다 AUM과 거래대금이 크고 변동성이 낮아...
4,국내 배당/방어,A161510,PLUS 고배당주,초기 편입,국내주식_섹터,Dividend,0.002300,-0.008195,-0.040048,0.311278,0.265005,5.459424e+10,PLUS 고배당주는 국내 방어형 위험자산 역할. 은행고배당보다 테마가 넓고 AUM이 큼.
5,국내 2차전지,A305720,KODEX 2차전지산업,관찰 후보,국내주식_섹터,Secondary Battery,0.004500,-0.069520,0.100515,0.226315,0.556559,2.140647e+11,2차전지는 반등 후보지만 최근 1개월 모멘텀이 약함. KODEX 2차전지산업은 AU...
6,국내 원자력,A434730,HANARO 원자력iSelect,관찰 후보,국내주식_섹터,Nuclear,0.004500,0.061540,0.246091,0.670263,0.609198,8.949205e+10,AI 전력 테마의 확장 후보. 단일 ETF이고 변동성이 높아 초기 핵심 편입보다는 ...
7,국내 로봇,A445290,KODEX 로봇액티브,관찰 후보,국내주식_섹터,Robotics,0.005689,0.204659,0.161727,0.834138,0.696477,1.158522e+11,모멘텀은 있으나 액티브 ETF이고 보수가 높아 후순위.
8,해외 AI 인프라,A487230,KODEX 미국AI전력핵심인프라,초기 편입,해외주식_섹터,AI,0.004500,0.124793,0.365391,0.547952,0.482784,5.887268e+10,미국 AI 전력 인프라 노출. 국내 AI 전력과 지역이 달라 보완 가능하나 해외섹터...
9,해외 반도체,A390390,KODEX 미국반도체,대체 후보,해외주식_섹터,Semiconductor,0.000900,0.263259,0.449820,0.737046,0.318408,4.425086e+10,KODEX 미국반도체는 보수가 낮고 3M 모멘텀이 강함. 다만 국내 AI/반도체와 ...


### 테마 선정 결론

초기 편입 테마는 국내 AI/반도체, AI 전력, 조선, 방산, 국내 배당, 해외 AI 인프라, 해외 배당, 금으로 제한한다. 2차전지, 원자력, 로봇, 미국 반도체, 달러, 엔, 원유는 후보군으로 유지하되 첫 달에는 제외한다.

이유는 세 가지다.

1. 삼성증권 리포트가 지적한 Spot Up, Vol Up 국면에서는 상승 추세를 따라가되 과도한 테마 확장을 피해야 한다.
2. 국내주식_섹터 15%, 해외주식_섹터 10% 상한이 있으므로 모든 유망 테마를 조금씩 담으면 각 ETF 비중이 너무 작아져 전략 효과가 희석된다.
3. ETF 구성종목과 지수 방법론을 아직 최종 확인하지 않았으므로, 1차 포트폴리오는 유동성·보수·테마 직접성이 명확한 ETF 위주로 구성한다.


## 5. 초기 포트폴리오 초안

초기 비중은 **위험자산 65%, 안전자산 35%** 로 둔다. 이는 규정상 허용되는 위험자산 70%보다 5%p 낮은 수준이다.

삼성증권 리포트 근거와 연결하면 다음과 같다.

- 리포트는 VKOSPI 60% 이상을 경계 구간으로 보고, 주가 상승과 변동성 상승이 동시에 나타나는 국면에서 타깃볼 투자자가 델타를 줄일 수 있다고 설명한다.
- 따라서 시장 상승을 완전히 포기하지는 않지만, 초기부터 위험자산을 70%까지 채우는 것은 리포트의 경고와 맞지 않는다.
- 변동성이 안정되면 65%에서 70%로 올리고, 반대로 VKOSPI가 재상승하면 55% 이하로 낮추는 방식이 투자철학과 운용방식의 일관성을 만든다.

포트폴리오 구성 원칙은 다음과 같다.

1. 국내 코어는 KOSPI200 중심, KOSDAQ150은 위성으로 제한한다.
2. 해외 코어는 S&P500과 NASDAQ100으로 구성하되 환헤지 ETF는 초기에는 쓰지 않는다.
3. 국내 섹터는 15% 상한을 꽉 채우되, 같은 테마 ETF는 중복 편입하지 않는다.
4. 해외 섹터는 8%만 사용해 10% 상한 대비 여유를 둔다.
5. 금은 위험자산으로 분류되지만 주식 급락 시 완충 역할을 기대해 5% 편입한다.
6. 안전자산은 CD/MMF/단기채 중심으로 두고, 장기채는 금리 하락 또는 위험회피 국면에 대비한 소량 옵션성 포지션으로만 편입한다.


In [7]:
initial_portfolio = pd.DataFrame([
    # Domestic equity index: 20%
    {'Ticker': 'A102110', 'Weight': 0.18, 'Role': '국내 코어: KOSPI200 대표 ETF'},
    {'Ticker': 'A229200', 'Weight': 0.02, 'Role': '국내 위성: KOSDAQ150, 관망 비중'},

    # Overseas equity index: 17%
    {'Ticker': 'A379800', 'Weight': 0.11, 'Role': '해외 코어: 미국 S&P500'},
    {'Ticker': 'A133690', 'Weight': 0.06, 'Role': '해외 성장: 미국 NASDAQ100'},

    # Domestic sectors: 15%
    {'Ticker': 'A395160', 'Weight': 0.04, 'Role': '국내 테마: AI/반도체 대표'},
    {'Ticker': 'A487240', 'Weight': 0.03, 'Role': '국내 테마: AI 전력 인프라'},
    {'Ticker': 'A466920', 'Weight': 0.02, 'Role': '국내 테마: 조선'},
    {'Ticker': 'A449450', 'Weight': 0.02, 'Role': '국내 테마: 방산'},
    {'Ticker': 'A434730', 'Weight': 0.02, 'Role': '국내 테마: 원자력/전력 보완'},
    {'Ticker': 'A445290', 'Weight': 0.02, 'Role': '국내 테마: 로봇/피지컬 AI'},

    # Overseas sectors: 8%
    {'Ticker': 'A487230', 'Weight': 0.04, 'Role': '해외 테마: 미국 AI 전력 인프라'},
    {'Ticker': 'A458730', 'Weight': 0.04, 'Role': '해외 방어형 테마: 미국 배당'},

    # FX and commodity: 5%
    {'Ticker': 'A411060', 'Weight': 0.05, 'Role': '비주식 헤지: 금 현물'},

    # Safe assets: 35%
    {'Ticker': 'A459580', 'Weight': 0.15, 'Role': '안전자산 핵심: CD 금리'},
    {'Ticker': 'A488770', 'Weight': 0.10, 'Role': '안전자산 핵심: 머니마켓'},
    {'Ticker': 'A157450', 'Weight': 0.05, 'Role': '안전자산: 단기통안채'},
    {'Ticker': 'A423160', 'Weight': 0.03, 'Role': '안전자산: KOFR 초단기 금리'},
    {'Ticker': 'A0061Z0', 'Weight': 0.02, 'Role': '안전자산: 단기특수은행채'},
])

portfolio_view = initial_portfolio.merge(
    valid[['Ticker', 'Name', 'Class1', 'Class2', 'Theme', 'AUM', 'Expense Ratio', 'Ret_1M', 'Ret_3M', 'Ret_6M', 'Vol_20D', 'TradingValue_20D']],
    on='Ticker',
    how='left'
)
portfolio_view['Weight_pct'] = portfolio_view['Weight'].map(lambda x: f'{x:.1%}')
portfolio_view[['Ticker', 'Name', 'Role', 'Class1', 'Class2', 'Theme', 'Weight_pct', 'Expense Ratio', 'Ret_1M', 'Ret_3M', 'Ret_6M', 'Vol_20D', 'TradingValue_20D']]


,Ticker,Name,Role,Class1,Class2,Theme,Weight_pct,Expense Ratio,Ret_1M,Ret_3M,Ret_6M,Vol_20D,TradingValue_20D
0,A102110,TIGER 200,국내 코어: KOSPI200 대표 ETF,위험,국내주식_지수,Broad Market,18.0%,0.000500,0.277688,0.470171,1.200160,0.553541,6.472338e+11
1,A229200,KODEX 코스닥150,"국내 위성: KOSDAQ150, 관망 비중",위험,국내주식_지수,Broad Market,2.0%,0.002500,-0.004830,-0.026575,0.260997,0.450267,5.537670e+11
2,A379800,KODEX 미국S&P500,해외 코어: 미국 S&P500,위험,해외주식_지수,Broad Market,11.0%,0.000700,0.082583,0.140117,0.158788,0.097403,4.533738e+11
3,A133690,TIGER 미국나스닥100,해외 성장: 미국 NASDAQ100,위험,해외주식_지수,Broad Market,6.0%,0.001000,0.140298,0.241480,0.230539,0.118525,1.677273e+11
4,A395160,KODEX AI반도체,국내 테마: AI/반도체 대표,위험,국내주식_섹터,AI; Semiconductor,4.0%,0.004500,0.378623,0.702111,1.634650,0.612449,2.259665e+11
5,A487240,KODEX AI전력핵심설비,국내 테마: AI 전력 인프라,위험,국내주식_섹터,AI,3.0%,0.003900,0.426574,0.818872,1.257972,0.841278,6.190343e+11
6,A466920,SOL 조선TOP3플러스,국내 테마: 조선,위험,국내주식_섹터,Shipbuilding,2.0%,0.004500,0.015174,0.033279,0.074971,0.434062,1.829848e+11
7,A449450,PLUS K방산,국내 테마: 방산,위험,국내주식_섹터,Defense,2.0%,0.004500,-0.076227,0.080272,0.349449,0.407595,8.050382e+10
8,A434730,HANARO 원자력iSelect,국내 테마: 원자력/전력 보완,위험,국내주식_섹터,Nuclear,2.0%,0.004500,0.061540,0.246091,0.670263,0.609198,8.949205e+10
9,A445290,KODEX 로봇액티브,국내 테마: 로봇/피지컬 AI,위험,국내주식_섹터,Robotics,2.0%,0.005689,0.204659,0.161727,0.834138,0.696477,1.158522e+11


In [8]:
validate_portfolio(initial_portfolio)

,구분,비중,상한,통과
0,Class1=위험,0.65,0.70,True
1,Class1=안전,0.35,1.00,True
2,Class2=국내주식_지수,0.20,0.30,True
3,Class2=국내주식_섹터,0.15,0.15,True
4,Class2=해외주식_지수,0.17,0.30,True
5,Class2=해외주식_섹터,0.08,0.10,True
6,Class2=FX 및 원자재,0.05,0.20,True
7,Class2=국내채권_종합,0.05,0.50,True
8,Class2=국내채권_회사채,0.02,0.30,True
9,Class2=해외채권_종합,0.00,0.50,True


In [9]:
allocation_by_class2 = (
    portfolio_view.groupby(['Class1', 'Class2'])['Weight']
    .sum()
    .sort_values(ascending=False)
    .map(lambda x: f'{x:.1%}')
)
allocation_by_class2

Class1  Class2     
안전      금리연계형/초단기채권    28.0%
위험      국내주식_지수        20.0%
        해외주식_지수        17.0%
        국내주식_섹터        15.0%
        해외주식_섹터         8.0%
안전      국내채권_종합         5.0%
위험      FX 및 원자재        5.0%
안전      국내채권_회사채        2.0%
Name: Weight, dtype: object

## 6. 리밸런싱 규칙

월 1회 정기 리밸런싱을 기본으로 하되, 삼성증권 리포트의 핵심 신호인 **VKOSPI 60% 경계선**과 **변동성 피크아웃 여부**를 위험자산 조절 기준으로 사용한다.

| 조건 | 조치 | 근거 |
|---|---|---|
| KOSPI200 추세 유지, VKOSPI 60 이하로 안정 | 위험자산 65% → 70% 확대 | 리포트상 변동성 피크아웃 이후 헤지성 선물매도 청산이 시장에 우호적일 수 있음 |
| KOSPI200 상승 지속, VKOSPI 60~70 | 전체 비중 유지, 테마 내부 교체만 수행 | Spot Up, Vol Up 국면에서는 추격매수보다 속도조절 우선 |
| VKOSPI 70 초과, KOSPI200 20일·60일선 유지, NAV/MDD 정상 | 위험자산 65% 유지, 신규 고베타 테마 증액 보류 | 고변동성 상승장에서는 기계적 매도 금지 |
| VKOSPI 70 초과 + (KOSPI200 20일선 이탈 또는 NAV/MDD 손실) | 위험자산 55% 이하로 축소, CD/MMF 확대 | 변동성 상승이 실제 손실로 전이될 때만 방어 |
| VKOSPI 80 초과 + (KOSPI200 추세 훼손 또는 NAV/MDD 방어 단계) | 위험자산 45~50%까지 축소 | 시스템 리스크에 준하는 방어 모드 |
| 개별 테마 ETF -12% 손실 또는 상대모멘텀 하위 30% 진입 | 해당 ETF 비중 절반 축소 | 테마 로테이션 실패 인정 |
| 동일 테마군 내 더 우수한 ETF 확인 | 다음 월 리밸런싱 때 교체 | 운용보수, 구성종목, 지수방법론 확인 후 반영 |

거래비용은 대회 공식 기준인 국내 ETF 거래수수료 0.1%, 제세금 없음으로 적용한다. 슬리피지는 공식 NAV 산정 비용이 아니므로 성과표에는 별도 차감하지 않고, 주문·체결 리스크 관리 항목으로만 분리한다. 백테스트에서는 최초 포트폴리오 설정 매매와 월간 리밸런싱, VKOSPI 고변동성+추세 훼손/NAV/MDD 신호 리밸런싱 때 실제 회전한 비중에 비례해 비용을 차감한다.



## 6-1. 보강안 반영: 위험 버퍼, 손실 통제, 사건 대응

`2026 DBGaps/GAPS_투자전략_보강_개선안_20260526.md`의 핵심은 기존 포트폴리오를 바꾸는 것이 아니라, 실제 운용 중 행동 규칙을 더 명확히 만드는 것이다. 따라서 초기 비중은 위험자산 65%, 안전자산 35%로 유지하되 아래 네 가지 실행 규칙을 추가한다.

### 위험자산 5%p 버퍼

위험자산 상한은 70%이지만 최초 비중은 65%로 시작한다. 남겨둔 5%p는 소극적 현금이 아니라 두 가지 목적의 운용 버퍼다.

| 구분 | 의미 | 행동 |
|---|---|---|
| 추가 편입 여력 | VKOSPI 안정과 KOSPI200 추세 유지가 확인될 때 위험자산을 늘릴 수 있는 공간 | VKOSPI 60 이하, KOSPI200 20일·60일선 상회 시 최대 70%까지 확대 |
| 한도 흡수 공간 | 위험자산 가격 상승으로 평가비중이 규정 상한에 접근하는 것을 막는 완충 | 위험자산 70~74%는 초과 허용, 74% 초과는 월간 조정, 76% 초과 또는 74% 초과+추세 훼손 때만 월중 조정 |

개별 ETF도 20% 상한까지 채우지 않는다. 현재 최대 비중은 TIGER 200 17%이며, 평가비중이 18%를 넘으면 신규 매수를 멈추되, 20~23% 구간은 평가이익 초과로 허용하고 23% 초과는 월간 조정, 25% 초과 또는 23% 초과+추세 훼손 때만 월중 조정한다.

### 현금성 자산의 역할

CD금리와 머니마켓 ETF는 단순 방어 편입이 아니라 리밸런싱 재원이다. 시장 추세가 유지되고 변동성이 안정되면 일부를 위험자산 확대에 사용하고, 변동성 또는 손실이 확대되면 고집중 위성 ETF의 매도대금을 다시 현금성 ETF에 보관한다.

| 시장 상황 | 현금성 자산 사용법 |
|---|---|
| 추세 유지, VKOSPI 안정 | CD/MMF 일부를 위험자산으로 이전 |
| VKOSPI 급등 + (KOSPI200 추세 훼손 또는 NAV/MDD 손실 확대) | 위성 위험자산 매도대금을 CD/MMF로 보관 |
| 조선·방산 부진, 원자력·2차전지 개선 | 즉시 추격하지 않고 현금성 자산에서 확인 기간 확보 |
| 유가발 물가 충격 | 장기채보다 CD/MMF 확대를 우선 |

### VKOSPI와 2차 확인지표

VKOSPI는 1차 신호이고, 유가·물가·금리는 2차 확인지표다. 같은 VKOSPI 상승이라도 공급충격인지 성장둔화인지에 따라 대응이 달라진다.

| 1차 신호 | 2차 확인지표 | 해석 | 행동 |
|---|---|---|---|
| VKOSPI 60 이하 | KOSPI200 추세 유지 + 유가·물가 안정 | 추세 지속과 위험 완화 | 위험자산 70%까지 확대 가능 |
| VKOSPI 60 이하 | 유가 또는 물가 재상승 | 금리위험 잔존 | 65% 유지, 즉시 확대 보류 |
| VKOSPI 70 초과 | KOSPI200 20일·60일선 유지, NAV/MDD 정상 | 고변동성 상승장 | 위험자산 65% 유지, 신규 고베타 테마 증액 보류 |
| VKOSPI 70 초과 | WTI 상승, CPI/PCE 상승, 장기채 약세, KOSPI200 추세 훼손 | 공급충격형 위험 현실화 | 조선·방산·AI전력 일부 축소, CD/MMF 확대, 장기채 확대 금지 |
| VKOSPI 70 초과 | 유가 안정, 시장금리 하락, 장기채 상승, KOSPI200 추세 훼손 | 성장둔화형 위험회피 | 위성 주식 축소, 장기채 소폭 확대 가능 |
| VKOSPI 80 초과 | KOSPI200 추세 훼손 또는 NAV/MDD 방어 단계 | 시스템성 방어 구간 | 위험자산 45~50%까지 축소 |

### NAV/MDD 손실 통제

개별 ETF 손실 규칙만으로는 동반 하락을 통제하기 어렵다. 따라서 전체 포트폴리오 NAV와 최대낙폭 기준을 추가한다.

| 단계 | 포트폴리오 기준 | 행동 | 목표 위험자산 |
|---|---|---|---:|
| 정상 | NAV 손실 -3% 미만, MDD -4% 미만 | 기존 전략 유지 | 65% |
| 주의 | NAV -3% 이하 또는 MDD -4% 이하 | 신규 위험 확대 중단, 손실 원인 재점검 | 최대 65% |
| 방어 전환 | NAV -5% 이하 또는 MDD -6% 이하 | 고집중·고변동 위성 ETF 5~10%p 축소, CD/MMF 확대 | 55% 내외 |
| 강제 방어 | NAV -7% 이하 또는 MDD -8% 이하 | 위성 축소 후 필요 시 지수 코어 일부 축소 | 45~50% |
| 재진입 | 방어 후 신호 개선 | 단기 반등 당일 추격 금지, 정기점검 또는 신호 2개 이상 확인 후 단계적 확대 | 55~65% |

축소 우선순위는 `SOL 조선TOP3플러스 → PLUS K방산 → KODEX AI전력핵심설비 → KODEX 코스닥150 → KODEX AI반도체 → TIGER 미국나스닥100 → TIGER 200/KODEX 미국S&P500` 순서다. 코어 지수는 가장 늦게 줄이고, 고집중 위성 ETF를 먼저 줄인다.



## 7. 채권/안전자산 추천

채권을 잘 모르는 상태라면, 이번 포트폴리오에서는 **금리 방향성 베팅보다 변동성 완충**을 우선하는 것이 좋다. 현재 데이터 기준으로도 CD/머니마켓/단기채는 20일 변동성이 매우 낮고, 장기채는 변동성이 훨씬 크다. 따라서 안전자산 35%는 아래처럼 역할을 나눠 가져간다.

| 역할 | ETF | 비중 | 이유 |
|---|---|---:|---|
| 현금성 방어 | KODEX CD금리액티브(합성) | 15% | 낮은 보수, 높은 거래대금, 낮은 변동성 |
| 현금성 방어 | KODEX 머니마켓액티브 | 10% | CD와 유사한 방어 역할, 분산 목적 |
| 단기채 | TIGER 단기통안채 | 5% | 금리 민감도가 낮고 채권형 자산 역할 명확 |
| 단기 크레딧 | RISE 단기특수은행채액티브 | 3% | 단기 우량 크레딧 성격, 낮은 변동성 |
| 금리 하락 옵션 | ACE 미국30년국채액티브(H) | 2% | 위험회피/금리 하락 시 방어 후보. 단, 변동성이 커서 소량만 편입 |

추천하지 않는 채권 포지션은 다음과 같다.

- 해외 하이일드: 이름은 채권이지만 위험자산 성격이 강해 안전자산 역할이 약하다.
- 미국 장기채 과대 비중: 금리 상승 시 손실이 커질 수 있으므로 지금은 2~5% 정도만 적합하다.
- 환노출 장기채: 채권 듀레이션과 환율 리스크가 동시에 생겨 초안 포트폴리오에는 복잡도가 크다.

결론적으로 안전자산의 80% 이상은 CD/MMF/단기채에 두고, 장기채는 시장이 급락하거나 금리 하락 기대가 커질 때 작동하는 보조 포지션으로만 사용한다.


## 7. 월간 운용보고서에서 점검할 항목

1. 전월 전망과 실제 성과의 차이
2. 포트폴리오 변경 사유
3. 자산군별 성과 기여도
4. 테마 ETF 선정/제외 사유
5. VKOSPI 변화에 따른 위험자산 비중 조정 근거
6. 익월 전망 및 유지/교체 후보

성과평가는 단순 수익률뿐 아니라 변동성, MDD, 위험자산 대비 초과수익, 동일 제약 조건의 정적 포트폴리오 대비 성과로 평가한다.

## 8. 투자계획서 문장 초안

본 포트폴리오는 2026년 6월~8월 운용기간 동안 고변동성 상승장에 대응하기 위해 설계된 타깃볼 기반 테마 로테이션 전략이다. 삼성증권 Derivatives Weekly 「Spot Up, Vol Up」(2026.05.08)은 최근 한국 시장을 주가 상승과 변동성 상승이 동시에 나타나는 국면으로 설명한다. 특히 KOSPI200이 강하게 상승하는 과정에서 VKOSPI가 60%를 상회했고, 이러한 환경에서는 타깃볼 투자자가 포트폴리오 변동성을 낮추기 위해 델타를 축소할 수 있다고 해석한다.

이 리포트를 근거로 본 전략은 위험자산을 완전히 회피하지 않되, 초기 위험자산 비중을 규정상 상한인 70%보다 낮은 65%로 설정한다. 시장의 주도 테마에는 참여하지만, 변동성이 높은 구간에서는 CD금리, 머니마켓, 단기채를 통해 포트폴리오의 변동성을 낮춘다. 향후 VKOSPI가 60 이하로 안정되고 KOSPI200 추세가 유지되면 위험자산 비중을 70%까지 확대하고, 반대로 VKOSPI가 70을 넘거나 지수 추세가 훼손되면 위험자산을 55% 이하로 축소한다.

테마 ETF는 universe에 명시된 Theme를 기준으로 AI/반도체, AI 전력 인프라, 조선, 방산, 배당, 금 등으로 구분한다. 같은 테마군 내 유사 ETF는 여러 개 편입하지 않고, 운용보수, AUM, 최근 거래대금, 가격 모멘텀, 변동성을 기준으로 대표 ETF 1개만 선정한다. ETF 구성종목과 지수 방법론은 최종 제출 전 추가 확인하여, 이미 편입한 ETF와 상위 구성종목이 과도하게 중복될 경우 교체한다.

안전자산은 단순히 채권을 많이 담는 방식이 아니라, 역할별로 나누어 구성한다. CD금리와 머니마켓 ETF는 현금성 방어자산으로 사용하고, 단기통안채와 단기특수은행채는 낮은 변동성의 이자수익 자산으로 편입한다. 미국 30년 국채(H)는 최근 성과가 부진하고 변동성이 높기 때문에 핵심 안전자산이 아니라, 위험회피 또는 금리 하락 국면에서 작동할 수 있는 소량의 옵션성 포지션으로만 사용한다.


## 9. 체크리스트 업데이트 (2026-05-25 기준)

기준 자료는 `universe.xlsx`, `factsheet/` 폴더의 운용보고서 53개, `ETF_운용보고서_추출_비교표.xlsx`, `VKOSPI.xlsx`, `코스피, 코스피200, 코스닥150.xlsx`이다.

| 체크 항목 | 상태 | 확인한 근거 | 남은 일 |
|---|---:|---|---|
| 운용보고서 PDF 파일명 정리 및 factsheet 폴더 이동 | [x] 완료 | `factsheet/`에 ETF 운용보고서 53개 저장. 2차전지/EV 후보 5개 추가 | 없음 |
| ETF별 구성종목/상위 10개 비중 | [x] 완료 | `ETF_운용보고서_추출_비교표.xlsx`의 `Holdings_Top10` 450행 | 최종 제출 전 핵심 ETF만 눈으로 재확인 |
| ETF별 업종/자산 비중 | [x] 완료 | `Sector_Weights` 204행 | 표에 넣을 핵심 ETF만 선별 |
| ETF 대표 선정/제외 표 | [x] 완료 | `Selection_Comparison` 49행. 2차전지/EV 및 추가 로테이션 후보 비교 추가 | 계획서 본문에는 핵심 비교만 압축 |
| 초기 편입 ETF 간 구성종목 중복 점검 | [x] 완료 | `Overlap_Initial`; 조선/방산 일부 중복 확인 | 조선·방산 비중 3% 유지 근거로 사용 |
| 테마 ETF 과다 중복 방지 | [x] 완료 | 국내 AI반도체, 국내 AI전력, 해외 AI전력, 조선, 방산, 배당을 각각 대표 1개 중심으로 분리 | 해외 반도체 ETF는 초기안에서 제외한 이유를 명시 |
| ETF 지수 방법론 | [x] 핵심완료 | 방법론 폴더의 공식 자료에서 AI반도체, AI전력, 미국AI전력, 조선, 방산, 고배당, 지수, 금, 단기통안채 방법론 확인 | KAP MMI/CD/KOFR, Dow Jones U.S. Dividend 100 세부 방법론 보강 |
| 테마 로테이션 전략 | [x] 완료 | 웹 자료와 로컬 PDF를 바탕으로 `AI 코어 + 순환/방어 바벨 + VKOSPI 오버레이` 규칙 추가 | 월말 데이터로 실제 교체 여부만 갱신 |
| 월별 행동 규칙 | [x] 완료 | 6월 초, 7월 초, 8월 초, 8월 말 행동 규칙 작성 | 월간 운용보고서에서 실제 성과에 맞춰 갱신 |
| VKOSPI 데이터 및 타깃볼 신호 | [x] 완료 | `VKOSPI.xlsx`; 2026-05-22 VKOSPI 66.97 | 6월 말/7월 말 수치로 업데이트 |
| KOSPI200 추세 판단 | [x] 완료 | `코스피, 코스피200, 코스닥150.xlsx`; KOSPI200 강세 추세 확인 | 월간 보고서마다 재계산 |
| 방법론/자료 폴더 반영 | [x] 완료 | GAPS_방법론_자료_보강_20260526.md, GAPS_방법론_자료_보강표_20260526.xlsx, 방법론/README.md, 자료/README.md 업데이트 | 제출용 4페이지에 핵심 문장만 압축 |
| 벤치마크 설정 | [-] 부분완료 | TIGER 200, KODEX CD금리액티브, KODEX 머니마켓액티브 가격 데이터는 수정주가 파일에 있음 | 실제 전략 수익률과 벤치마크 수익률 비교표/그래프 생성 필요 |
| 리스크 관리 문장 | [x] 완료 | VKOSPI 60/70/80 기준, ETF 20% 상한, 세부자산군 상한 반영 | 손절/축소 규칙을 최종 문장으로 압축 |
| 채권/안전자산 설명 | [x] 완료 | CD금리, 머니마켓, 단기채, 단기특수은행채, 미국30년국채(H) 역할 구분 | 금리 시나리오 문장만 조금 더 다듬기 |
| 제출용 4페이지 구성 | [-] 부분완료 | 투자철학, 자산군 전망, ETF 선정 근거, 포트폴리오/리스크관리 구조 작성 | Word 문서용 문장 압축과 표 배치 필요 |

현재 가장 중요한 미완료는 **벤치마크 성과 비교 숫자화**, **지수 ETF 포함 상위 종목 중복 계산**, **현금성 자산 세부 방법론 보강**이다. 핵심 테마 ETF의 공식 방법론은 확인했으므로, 이제는 제출용 4페이지에 넣을 문장 압축과 성과 비교표 작성이 우선이다.




### 보강안 추가 반영 체크

| 체크 항목 | 상태 | 확인한 근거 | 남은 일 |
|---|---:|---|---|
| 위험자산 5%p 버퍼 설명 | [x] 완료 | 위험자산 65%와 상한 70% 사이의 운용 버퍼 문장 추가 | 제출용 문장 압축 |
| 개별 ETF 평가비중 경보선 | [x] 완료 | 18%, 19%, 19.5% 기준 추가 | 실제 주문 전 평가비중 재계산 |
| 현금성 자산의 리밸런싱 재원 정의 | [x] 완료 | CD금리·머니마켓 역할 표 추가 | KAP MMI/CD 방법론 보강 시 문장 업데이트 |
| VKOSPI 2차 확인지표 | [x] 완료 | 유가·물가·금리별 공급충격/성장둔화 구분 추가 | WTI 또는 원유 관측 ETF 데이터 확인 |
| NAV/MDD 손실 통제 | [x] 완료 | NAV -3/-5/-7%, MDD -4/-6/-8% 기준 추가 | 운용 시작 후 NAV 계산 코드 연결 |
| 거시사건 대응 프로토콜 | [x] 완료 | 공급충격형·성장둔화형·긴장완화형 행동 규칙 정리 | 대회 원문 규정의 거래 가능 조건 재확인 |
| 공식 지수 방법론 반영 | [x] 완료 | 핵심 ETF의 선정 방식, 가중 방식, 리밸런싱 주기, 집중도 위험을 전략에 연결 | CD/MMF와 미국배당 세부 방법론만 추가 확인 |
| 시장전망 자료 반영 | [x] 완료 | 삼성증권, 대신증권, 유안타증권 리포트를 전략 근거에 연결 | 기사형 자료는 보조 근거로만 사용 |



## 10. 추가 필요자료 및 남은 작업

### 추가 자료 필요 여부

기존 요청 목록의 운용보고서는 모두 충족됐다. `A381180 TIGER 미국필라델피아반도체나스닥` 운용보고서도 `factsheet/` 폴더에 추가했고, 비교 엑셀에도 반영했다.

다만 운용보고서보다 더 필요한 자료는 아래와 같다.

| 우선순위 | 필요한 자료 | 쓰는 위치 | 이유 |
|---:|---|---|---|
| 1 | 핵심 ETF의 공식 지수 방법론 | ETF 선정 근거 | 운용보고서에는 기초지수 설명만 있고, 시총가중/동일가중/리밸런싱 주기가 부족한 경우가 있음 |
| 2 | 벤치마크 수익률 계산표 | 성과평가 기준 | 가격 데이터는 이미 있음. KOSPI200 100%, TIGER 200 100%, TIGER 200 65%+CD 35% 리밸런싱 없음, 초기 65:35 리밸런싱 없음, 계획서형 Target Vol Overlay 비교표를 반영함 |
| 3 | 핵심 ETF별 추적오차/괴리율 또는 최근 거래대금 | ETF 선정 근거 | 운용보수만으로는 대표 ETF 선정 논리가 약할 수 있음 |
| 4 | 최종 편입/제외 ETF 압축표 | 상품별 전략 | 53개 ETF를 모두 설명하면 산만하므로 대표/비교/제외만 보여줘야 함 |
| 5 | 6월 이후 월말 성과 데이터 | 월간 운용보고서 | 실제 운용 후 전망과 성과 차이를 설명하는 데 필요 |

### 하반기 로테이션 후보로 추가 검토할 ETF

사용자 메모의 하반기 후보인 2차전지·신재생/에너지 전환·방어 섹터는 현재 초기 포트폴리오에는 넣지 않았다. 다만 7월 이후 신호가 살아나면 아래 ETF를 관찰 후보로 둘 수 있다.

| 후보 테마 | Ticker | ETF명 | 판단 |
|---|---|---|---|
| 2차전지 대표 | A305720 | KODEX 2차전지산업 | 운용보고서와 방법론 확인 완료. AUM/대표성 기준 7월 교체 후보 1순위 |
| 2차전지 대체 | A305540 | TIGER 2차전지테마 | 운용보고서 확인 완료. KODEX 2차전지산업과 유사한 대체 비교 후보 |
| 2차전지 소재 | A462010 | TIGER 2차전지소재Fn | 운용보고서 확인 완료. 소재 집중형 고베타 후보로 편입 시 1~2% 제한 |
| 원자력/전력 | A434730 | HANARO 원자력iSelect | AI 전력 테마 확장 후보. 이미 운용보고서 확보 |
| 로봇/피지컬 AI | A445290 | KODEX 로봇액티브 | 운용보고서 확인 완료. AI 현실 자동화 확장 후보이나 액티브·보수 0.50%라 1~2% 제한 |
| 에너지 | A218420 | KODEX 미국S&P500에너지(합성) | 운용보고서 확인 완료. 공급충격 헤지 후보이나 합성형·AUM 작음 |
| 원유 관측 | A261220 | KODEX WTI원유선물(H) | 운용보고서 확인 완료. 최초 편입보다 VKOSPI 2차 확인지표로 사용 |
| 헬스케어 | A453640 | KODEX 미국S&P500헬스케어 | 운용보고서 확인 완료. 성장둔화형 위험회피 후보 |
| 필수소비재 | A453630 | KODEX 미국S&P500필수소비재 | 운용보고서 확인 완료. 침체 전환 방어 후보이나 AUM이 작아 보조 후보 |
| 리츠 | A182480 | TIGER 미국MSCI리츠(합성 H) | 운용보고서 확인 완료. 금리하락형 후보이나 합성 H 구조 확인 필요 |
| 철강소재 | A139240 | TIGER 200 철강소재 | 운용보고서 확인 완료. 조선·인프라 확장 후보이나 AUM 작고 집중도 높음 |
| 경기소비재 | A139290 | TIGER 200 경기소비재 | 운용보고서 확인 완료. 경기회복 후보이나 자동차 집중도 높음 |

### 지금 노트북에서 정리해야 할 부분

1. 관찰 후보로만 적어둔 2차전지, 로봇, 중국 EV, 달러, 엔, 원유 ETF는 최종 계획서에 쓸지 결정해야 한다. 쓰지 않을 것이라면 본문에서는 삭제하는 편이 낫다.
2. `KODEX AI전력핵심설비`와 `HANARO 원자력iSelect`는 전력 인프라 관점에서 일부 성격이 겹치므로, 초기안에서는 AI전력 ETF를 유지하고 원자력은 교체 후보로 두는 논리를 명확히 해야 한다.
3. `PLUS 고배당주`, `KODEX 은행`, `TIGER 은행고배당플러스TOP10`은 모두 방어/배당 성격이 있지만, 은행 단일업종 베팅을 피하기 위해 `PLUS 고배당주`를 유지한다는 논리를 본문에 넣어야 한다.
4. 해외 배당 ETF는 `TIGER 미국배당다우존스`를 대표로 두고, `SOL 미국배당다우존스(H)`, `SOL 미국배당다우존스TR`, `TIGER 미국S&P500배당귀족`은 비교 후보로만 처리하는 것이 깔끔하다.
5. 벤치마크는 `KOSPI200 100%`, `TIGER 200 100%`, `TIGER 200 65%+CD 35% 리밸런싱 없음`, `초기 65:35 리밸런싱 없음`, `계획서형 Target Vol Overlay`로 계산한다.

### 다음 작업 순서

1. 핵심 ETF 10~12개만 골라 지수 방법론 문장을 보강한다.
2. 노트북에 벤치마크 수익률 비교 코드를 추가한다.
3. 최종 포트폴리오 표와 선정/제외 표를 계획서 4페이지 형식에 맞게 압축한다.
4. 삼성증권 리포트 근거, VKOSPI 신호, KOSPI200 추세 판단, 테마 로테이션 규칙을 한 문단으로 연결한다.




## 11. 운용보고서 PDF 분석 결과

`factsheet/` 폴더의 ETF 운용보고서 53개를 기준으로 ETF별 기초지수/방법론 설명, 상위 10개 구성종목, 업종/자산비중을 추출했다. 결과 파일은 `ETF_운용보고서_추출_비교표.xlsx`로 저장했다.

엑셀 구성은 다음과 같다.

| 시트 | 내용 | 행 수 |
|---|---|---:|
| `ETF_Summary` | ETF별 요약, 상위 종목 집중도, 보수/AUM 출처 | 53 |
| `Holdings_Top10` | ETF별 상위 보유종목 | 450 |
| `Sector_Weights` | ETF별 업종/자산 비중 | 204 |
| `Overlap_Initial` | 초기 편입 ETF 간 구성종목 중복 | 28 |
| `Selection_Comparison` | 대표 ETF 선정/비교/제외 근거 | 49 |
| `Rotation_Candidates` | 7월 이후 신호별 교체 후보 | 8 |

핵심 결론은 다음과 같다.

1. 초기 편입 ETF 간 구성종목 중복은 크지 않다. 자동 추출 기준으로는 `SOL 조선TOP3플러스`와 `PLUS K방산` 사이에 `한화오션`, `STX엔진`이 겹친다.
2. 테마 ETF 내부 집중도는 높다. `KODEX AI전력핵심설비`, `SOL 조선TOP3플러스`, `PLUS K방산`, `KODEX 미국AI전력핵심인프라`는 상위 5개 종목 비중이 높기 때문에 각 ETF 비중을 3~4% 수준으로 제한한다.
3. `TIGER 미국필라델피아반도체나스닥`은 SOX 대표 ETF이고 유동성도 좋지만, 국내 AI반도체 및 미국 나스닥100과 반도체/빅테크 베타가 겹쳐 초기 포트폴리오에서는 제외한다.
4. `PLUS 고배당주`와 `TIGER 미국배당다우존스`는 상위 10개 집중도가 상대적으로 낮아 방어형 위험자산 역할에 적합하다.
5. `ACE KRX금현물`은 금 현물 100%에 가까운 구조라 주식 테마와 구성종목 중복이 없다.
6. `HANARO 원자력iSelect`, `KODEX 은행`, `SOL 초단기채권액티브`, `RISE 머니마켓액티브`, `TIGER 미국달러단기채권액티브` 등 새로 추가한 후보는 비교표에 반영했지만, 초기 포트폴리오에는 필요한 경우에만 교체 후보로 사용한다.

이 결과는 투자계획서의 **ETF 선정/제외 근거**와 **리스크 관리** 파트에 사용한다.



## 11-1. 공식 지수 방법론 반영 결과

방법론 폴더의 공식 지수 자료를 읽어 핵심 ETF 선정 근거를 보강했다. 운용보고서가 현재 무엇을 들고 있는지를 보여준다면, 지수 방법론은 왜 그런 종목들이 ETF에 들어오는지를 설명한다.

| ETF/자산 | 방법론 핵심 | 포트폴리오 반영 |
|---|---|---|
| KODEX AI반도체 | AI 반도체 키워드를 증권사 보고서와 공시에서 추출하고, 상위 2개 종목 비중이 큰 구조 | AI/HBM 직접성은 높지만 고집중형이므로 4% 제한 |
| KODEX AI전력핵심설비 | 전력기기, 전선, 데이터센터 에너지효율, ESS 등 AI 전력 설비를 키워드와 정성평가로 선별 | 국내 AI 전력 수요 노출로 3% 편입, 집중도 때문에 증액 신중 |
| KODEX 미국AI전력핵심인프라 | 미국 AI 전력 인프라 관련 10개 종목에 시가총액 70%와 키워드 순위 30%를 결합 | 해외 AI 인프라 분산 역할, 4% 제한 |
| SOL 조선TOP3플러스 | 조선 대표 3개 종목을 각 25% 수준으로 반영하고 기자재 종목 추가 | 순환 위성 3%, 손실 통제 시 우선 축소 후보 |
| PLUS K방산 | 보고서·사업보고서 키워드 유사도, 시가총액, 업종 가중치를 결합해 10개 종목 선정 | 정책·수출 테마로 3% 편입, 실제 구성종목 확인 병행 |
| PLUS 고배당주 | ROE, ROA 안정성, 예상 배당수익률, 유동성을 반영해 30개 종목 선정 | 은행 단일업종보다 분산된 방어형 위험자산 |
| TIGER 200 / KODEX 미국S&P500 / TIGER 미국나스닥100 | 시장 대표 지수를 시가총액 중심으로 추종 | 분산 코어지만 삼성전자, SK하이닉스, NVIDIA 등 간접 AI 노출을 함께 계산 |
| ACE KRX금현물 | KRX 금 현물 가격 기반 지수 | 선물 롤오버가 아닌 현물 기반 물가·지정학 헤지 |
| TIGER 단기통안채 | 단기 통안증권 바스켓을 월별 교체 | 낮은 듀레이션의 안정축 |

계획서 문장으로는 다음처럼 압축한다.

> ETF 선정은 단순 테마 선호가 아니라 공식 지수 방법론을 기준으로 한다. AI반도체와 AI전력 ETF는 키워드와 공시·보고서 기반 선별 구조를 통해 테마 직접성이 높지만, 소수 종목 집중도가 높으므로 3~4% 위성 비중으로 제한한다. 반대로 TIGER 200, S&P500, 나스닥100은 분산 코어로 쓰되, 이들 안에도 AI·반도체 대형주가 포함되어 있으므로 직접 AI ETF와 합산해 집중도를 관리한다.

남은 보강은 KAP MMI/CD/KOFR 방법론과 Dow Jones U.S. Dividend 100 세부 방법론이다. 이는 현금성 자산과 미국 배당 ETF의 설명을 더 정확히 만들기 위한 보완 과제이며, 핵심 테마 ETF 선정에는 큰 영향을 주지 않는다.


## 11-2. 2차전지 운용보고서 추가 반영

2차전지/EV 후보 운용보고서 5개와 FnGuide 2차전지 산업 지수 방법론을 추가 확인했다. 결론은 초기 편입이 아니라 7월 이후 상대모멘텀 개선 시 국내 섹터 내부 1~3% 교체 후보로 관리하는 것이다.

| 후보 | 성격 | Top5 | Top10 | 판단 |
|---|---|---:|---:|---|
| A305720 KODEX 2차전지산업 | 국내 2차전지 밸류체인 대표 | 70.66% | 92.27% | AUM과 대표성이 가장 좋아 7월 교체 후보 1순위 |
| A305540 TIGER 2차전지테마 | 국내 2차전지 테마 대체 | 68.70% | 91.47% | KODEX와 유사한 비교 후보 |
| A462010 TIGER 2차전지소재Fn | 소재 집중형 고베타 | 87.46% | 96.28% | 반등 탄력은 크지만 집중도가 매우 높아 1~2% 제한 후보 |
| A371460 TIGER 차이나전기차SOLACTIVE | 중국 EV·배터리 | 49.34% | 65.09% | 중국 정책·환율 리스크가 달라 선택 관찰 |
| A457480 ACE 테슬라밸류체인액티브 | 테슬라·EV 밸류체인 | 44.29% | 63.07% | 테슬라 24.33% 집중으로 2차전지 직접 대체보다는 보조 후보 |

`KODEX 2차전지산업`의 기초지수인 FnGuide 2차전지 산업 지수는 2차전지 Value Chain의 원재료, 장비, 부품, 제조 관련 키워드 스코어 상위 25종목을 선정하고, 스코어와 유동시가총액을 함께 반영한다. 방법론상 최대 종목 20%, 나머지 15% 실링 구조지만 실제 ETF Top10 비중이 90%를 넘기 때문에 고집중 테마로 관리한다.

따라서 2차전지는 조선·방산의 상대성과가 약해지고 2차전지의 1개월·3개월 상대모멘텀이 동시에 개선될 때만 1~3%p 교체를 검토한다. VKOSPI 70 초과 단독 구간에서는 신규 증액을 보류하고, VKOSPI 70 초과에 KOSPI200 추세 훼손 또는 NAV 방어 전환이 동반되면 신규 편입하지 않는다.


## 11-3. 추가 로테이션 후보 반영

2차전지 외에도 로봇, 에너지, 원유, 헬스케어, 필수소비재, 철강소재, 경기소비재, 리츠 운용보고서를 추가 반영했다. 이들은 초기 편입 대상이 아니라 7월 이후 신호별 교체 후보 또는 리스크 관측지표다.

| 후보 | 역할 | Top5 | Top10 | 판단 |
|---|---|---:|---:|---|
| A445290 KODEX 로봇액티브 | 로봇/피지컬 AI 확장 | 38.41% | 64.04% | AI 현실 자동화 확장 후보. 액티브·보수 0.50%라 1~2% 제한 |
| A218420 KODEX 미국S&P500에너지(합성) | 공급충격 헤지 | 100.00% | 100.00% | 유가 상승 수혜 후보이나 합성형·AUM 작음 |
| A261220 KODEX WTI원유선물(H) | 유가 관측지표 | 100.00% | 100.00% | 최초 편입보다 VKOSPI 2차 확인지표로 사용 |
| A453640 KODEX 미국S&P500헬스케어 | 성장둔화 방어 | 41.06% | 56.93% | 성장둔화형 위험회피 시 후보 |
| A453630 KODEX 미국S&P500필수소비재 | 침체 전환 방어 | 40.24% | 62.28% | 완전 방어 후보이나 AUM이 작아 보조 후보 |
| A139240 TIGER 200 철강소재 | 소재/리플레이션 | 87.09% | 99.36% | 조선·인프라 확장 후보이나 후순위 |
| A139290 TIGER 200 경기소비재 | 경기 회복 | 74.54% | 85.10% | 자동차 중심 경기회복 후보 |
| A182480 TIGER 미국MSCI리츠(합성 H) | 금리하락 후보 | 40.03% | 56.66% | 장기금리 하락 시 후보 |

운용 규칙은 단순하다. AI 확장이 계속되면 로봇, 공급충격이면 에너지와 원유 관측, 성장둔화면 헬스케어·필수소비재, 금리하락이면 리츠를 본다. 다만 이 후보들은 대부분 AUM이 작거나 합성/선물/액티브 구조이므로 초기 포트폴리오에 바로 넣지 않는다.


### factsheet 폴더 정리

ETF 운용보고서 PDF 원본은 `factsheet/` 폴더로 이동했다. 전략 근거 리포트인 삼성증권 Derivatives Weekly PDF는 ETF 운용보고서가 아니므로 DBgaps 루트에 남겨둔다.


In [10]:
PDF_EXTRACT_PATH = DATA_DIR / 'ETF_운용보고서_추출_비교표.xlsx'

pdf_summary = pd.read_excel(PDF_EXTRACT_PATH, sheet_name='ETF_Summary')
pdf_holdings = pd.read_excel(PDF_EXTRACT_PATH, sheet_name='Holdings_Top10')
pdf_sectors = pd.read_excel(PDF_EXTRACT_PATH, sheet_name='Sector_Weights')
pdf_overlap = pd.read_excel(PDF_EXTRACT_PATH, sheet_name='Overlap_Initial')
pdf_selection = pd.read_excel(PDF_EXTRACT_PATH, sheet_name='Selection_Comparison')

initial_theme_tickers = ['A395160', 'A487240', 'A466920', 'A449450', 'A161510', 'A487230', 'A458730', 'A411060']

pdf_summary[pdf_summary['Ticker'].isin(initial_theme_tickers)][
    ['Ticker', 'Name', 'Top1', 'Top1_Weight', 'Top5_Weight', 'Top10_Weight', 'Top10_Holdings']
].sort_values('Ticker')


,Ticker,Name,Top1,Top1_Weight,Top5_Weight,Top10_Weight,Top10_Holdings
8,A161510,PLUS 고배당주,현대차,7.10,27.85,49.57,현대차 7.10%; NH투자증권 5.75%; 기아 5.20%; DB손해보험 5.04...
20,A395160,KODEX AI반도체,SK하이닉스,23.46,69.70,85.30,SK하이닉스 23.46%; 삼성전자 21.60%; 한미반도체 12.98%; 이수페타...
23,A411060,ACE KRX금현물,040200 금 현물 99.99_1Kg,100.00,100.00,100.00,040200 금 현물 99.99_1Kg 100.00%
30,A449450,PLUS K방산,한화에어로스페이스,19.43,84.42,99.76,한화에어로스페이스 19.43%; 현대로템 18.61%; LIG디펜스앤에어로스페이스 ...
39,A458730,TIGER 미국배당다우존스,Texas Instruments In,5.36,23.05,42.06,Texas Instruments In 5.36%; UnitedHealth Group...
43,A466920,SOL 조선TOP3플러스,HD현대중공업,26.76,91.51,98.72,HD현대중공업 26.76%; 삼성중공업 24.54%; 한화오션 21.24%; HD한...
47,A487230,KODEX 미국AI전력핵심인프라,GE Vernova LLC,18.15,73.69,100.01,GE Vernova LLC 18.15%; Vertiv Holdings Co 17.0...
48,A487240,KODEX AI전력핵심설비,LS ELECTRIC,26.05,81.61,97.70,LS ELECTRIC 26.05%; 효성중공업 19.07%; HD현대일렉트릭 14....


In [11]:
# 초기 편입 ETF 간 구성종목 중복 확인
pdf_overlap[pdf_overlap['Common_Count'] > 0]


,ETF_A,ETF_B,Common_Count,Overlap_MinWeight_Sum,Common_Holdings
0,A466920,A449450,2,15.22,"STX엔진, 한화오션"


### PDF 분석을 반영한 ETF 선정 문장

- `KODEX AI반도체`는 SK하이닉스, 삼성전자, 한미반도체 등 AI 반도체 대형주와 핵심 소부장을 함께 보유해 AI 반도체 테마 직접성이 높다. 단, 상위 10개 비중이 높기 때문에 4%로 제한한다.
- `KODEX AI전력핵심설비`는 LS ELECTRIC, 효성중공업, HD현대일렉트릭 등 AI 데이터센터 전력 인프라 수혜주에 집중되어 있어 전략 테마와 부합한다. 상위 5개 집중도가 높아 3%만 편입한다.
- `SOL 조선TOP3플러스`는 조선 대표 3종목과 밸류체인 종목을 담는 구조로 테마 순환 효과가 강하다. 집중도가 높아 3% 편입한다.
- `PLUS K방산`은 방산 키워드 기반 종목 선정과 유동시가총액 가중 방식으로 구성되어 K방산 테마를 직접 반영한다. 조선 ETF와 일부 종목이 겹치므로 3%로 제한한다.
- `PLUS 고배당주`와 `TIGER 미국배당다우존스`는 테마 공격성을 낮추는 완충 역할이다.
- `ACE KRX금현물`은 주식 ETF와 구성종목 중복이 없고, 변동성 확대 시 방어자산 역할을 기대할 수 있다.


## 12. VKOSPI 데이터 필요 여부

VKOSPI는 필요하다. 이 전략은 단순 테마 로테이션이 아니라 **Target Vol Overlay** 전략이기 때문에, 월별 위험자산 비중 조절에 VKOSPI가 들어가야 한다.

필요한 데이터는 다음과 같다.

| 데이터 | 용도 | 최소 주기 |
|---|---|---|
| VKOSPI 종가 | 위험자산 비중 조절 신호 | 일간 또는 월말 |
| KOSPI200 종가 | 시장 추세 판단 | 일간 |
| VKOSPI 20영업일 변화율/이동평균 | 변동성 피크아웃 확인 | 일간 |

DataGuide에서 찾을 때는 아래 키워드로 검색한다.

- `VKOSPI`
- `V-KOSPI 200`
- `KOSPI200 변동성지수`
- `KOSPI 200 Volatility Index`
- `파생 및 기타지수`

DataGuide에서 바로 안 보이면 KRX 정보데이터시스템에서 받을 수 있다. KRX Data Marketplace에는 `지수 > 파생 및 기타지수 > 개별지수 시세 추이` 메뉴가 있고, V-KOSPI 200은 KOSPI200 변동성지수로 분류된다.

전략에서는 일단 아래 규칙으로 사용한다.

| VKOSPI 상태 | 판단 | 포트폴리오 조치 |
|---|---|---|
| 60 이하로 하락 안정 | 변동성 피크아웃 | 위험자산 65%에서 70%까지 확대 가능 |
| 60~70 | 고변동성 상승장 지속 | 전체 비중 유지, 테마 내부 교체만 수행 |
| 70 초과 | 과열/헤지 압력 확대 | 위험자산 55% 이하로 축소 |
| 80 초과 | 시스템 리스크성 변동성 | 위험자산 45~50%까지 축소 |

VKOSPI 데이터를 못 구하면 임시 대체값으로 KOSPI200 20영업일 실현변동성을 사용할 수 있지만, 투자계획서에서는 VKOSPI를 우선 신호로 쓰는 것이 더 논리적이다.


In [12]:
def load_vkospi_xlsx(path=VKOSPI_PATH):
    raw = pd.read_excel(path, sheet_name='Sheet1', header=None)
    df = raw.iloc[14:, :2].copy()
    df.columns = ['Date', 'VKOSPI']
    df['Date'] = pd.to_datetime(df['Date'])
    df['VKOSPI'] = pd.to_numeric(df['VKOSPI'], errors='coerce')
    return df.dropna().set_index('Date').sort_index()

vkospi = load_vkospi_xlsx()
vkospi_latest_date = vkospi.index.max()
vkospi_latest = float(vkospi.loc[vkospi_latest_date, 'VKOSPI'])
vkospi_20d_mean = float(vkospi['VKOSPI'].tail(20).mean())
vkospi_20d_change = float(vkospi['VKOSPI'].iloc[-1] - vkospi['VKOSPI'].iloc[-21])

if vkospi_latest <= 60:
    vol_signal = '변동성 안정: 위험자산 70%까지 확대 가능'
elif vkospi_latest <= 70:
    vol_signal = '고변동성 유지: 위험자산 65% 유지, 테마 내부 교체 중심'
elif vkospi_latest <= 80:
    vol_signal = '과열/헤지 압력 확대: 위험자산 55% 이하 축소'
else:
    vol_signal = '시스템 리스크성 변동성: 위험자산 45~50%까지 축소'

pd.DataFrame([{
    'LatestDate': vkospi_latest_date.date(),
    'VKOSPI': vkospi_latest,
    'VKOSPI_20D_Mean': vkospi_20d_mean,
    'VKOSPI_20D_Change': vkospi_20d_change,
    'Signal': vol_signal,
}])


,LatestDate,VKOSPI,VKOSPI_20D_Mean,VKOSPI_20D_Change,Signal
0,2026-05-22,66.97,64.036,14.85,"고변동성 유지: 위험자산 65% 유지, 테마 내부 교체 중심"


현재 `VKOSPI.xlsx` 기준 최신 관측치는 2026년 5월 22일의 66.97이다. 이는 60을 넘는 고변동성 구간이지만 70 이상에서는 내려온 상태다. 따라서 초기 포트폴리오는 위험자산 65%를 유지하고, VKOSPI가 60 이하로 안정되는지를 6월 첫 리밸런싱의 핵심 조건으로 둔다.


## 13. KOSPI200 추세 판단

`코스피, 코스피200, 코스닥150.xlsx`를 추가로 받아 시장 추세를 확인한다. 전략 논리상 핵심 추세 지표는 KOSPI가 아니라 **KOSPI200**이다. 이유는 VKOSPI가 KOSPI200 옵션 기반 변동성지수이고, 삼성증권 리포트도 KOSPI200과 VKOSPI의 동행을 중심으로 설명하기 때문이다.

현재 데이터 기준 KOSPI200은 20일/60일 이동평균 위에 있고, 1개월·3개월·6개월·12개월 수익률이 모두 양수다. 따라서 가격 추세는 강세로 판단한다. 다만 VKOSPI가 60을 넘는 고변동성 구간이므로 위험자산을 70%까지 바로 확대하지 않고 65%로 시작한다.


In [13]:
def load_market_index_xlsx(path=MARKET_INDEX_PATH):
    raw = pd.read_excel(path, sheet_name='Sheet1', header=None)
    names = raw.iloc[9, 1:4].tolist()
    df = raw.iloc[14:, :4].copy()
    df.columns = ['Date'] + names
    df['Date'] = pd.to_datetime(df['Date'])
    for col in names:
        df[col] = pd.to_numeric(df[col], errors='coerce')
    return df.dropna().set_index('Date').sort_index()

market_index = load_market_index_xlsx()

def index_stats(series):
    return {
        'Latest': float(series.iloc[-1]),
        'Ret_1M': series.iloc[-1] / series.iloc[-22] - 1,
        'Ret_3M': series.iloc[-1] / series.iloc[-64] - 1,
        'Ret_6M': series.iloc[-1] / series.iloc[-127] - 1,
        'Ret_12M': series.iloc[-1] / series.iloc[-253] - 1,
        'MA20': float(series.tail(20).mean()),
        'MA60': float(series.tail(60).mean()),
        'Above_MA20': bool(series.iloc[-1] > series.tail(20).mean()),
        'Above_MA60': bool(series.iloc[-1] > series.tail(60).mean()),
    }

market_signal = pd.DataFrame([
    {'Index': col, **index_stats(market_index[col])}
    for col in market_index.columns
])
market_signal


,Index,Latest,Ret_1M,Ret_3M,Ret_6M,Ret_12M,MA20,MA60,Above_MA20,Above_MA60
0,코스피,7847.71,0.228418,0.382308,0.956269,2.009864,7262.5400,6229.192167,True,True
1,코스피 200,1226.03,0.274115,0.459143,1.176011,2.532413,1116.0985,940.113500,True,True
2,코스닥 150,1991.85,-0.004657,-0.024177,0.261631,0.702509,1977.9880,1958.599000,True,True


In [14]:
kospi200_stats = index_stats(market_index['코스피 200'])

if kospi200_stats['Ret_12M'] > 0 and kospi200_stats['Above_MA60']:
    trend_signal = 'KOSPI200 강세 추세'
else:
    trend_signal = 'KOSPI200 추세 둔화/약세'

combined_signal = pd.DataFrame([{
    'Date': market_index.index.max().date(),
    'KOSPI200': kospi200_stats['Latest'],
    'KOSPI200_1M': kospi200_stats['Ret_1M'],
    'KOSPI200_3M': kospi200_stats['Ret_3M'],
    'KOSPI200_12M': kospi200_stats['Ret_12M'],
    'Trend_Signal': trend_signal,
    'VKOSPI': vkospi_latest,
    'Vol_Signal': vol_signal,
    'Portfolio_Action': '위험자산 65% 유지' if '고변동성 유지' in vol_signal and '강세' in trend_signal else vol_signal,
}])
combined_signal


,Date,KOSPI200,KOSPI200_1M,KOSPI200_3M,KOSPI200_12M,Trend_Signal,VKOSPI,Vol_Signal,Portfolio_Action
0,2026-05-22,1226.03,0.274115,0.459143,2.532413,KOSPI200 강세 추세,66.97,"고변동성 유지: 위험자산 65% 유지, 테마 내부 교체 중심",위험자산 65% 유지


## 14. 테마 로테이션 전략: AI 코어 + 순환/방어 바벨

### 외부 자료에서 확인한 핵심 방향

1. 경기 사이클별 섹터 로테이션의 기본 틀은 유지한다. Fidelity는 회복 초기에는 경기민감주와 기술주, 확장기에는 기술주와 산업재, 후반부에는 에너지·소재, 둔화/침체기에는 헬스케어·필수소비재·유틸리티 같은 방어 섹터가 상대적으로 강할 수 있다고 설명한다.
2. 2026년 글로벌 자료는 시장 리더십이 빅테크 단일 축에서 더 넓은 가치·순환·방어 섹터로 확산되는 방향을 보여준다. BofA 설문을 인용한 Funds Society 자료는 2026년 2월 기준 에너지, 소재, 필수소비재 비중이 늘고 미국 기술주 및 달러 비중이 줄었다고 설명한다.
3. 다만 2026년 5월 BofA 설문을 인용한 Axios 자료는 글로벌 매니저의 주식 비중이 크게 높아졌고, 동시에 인플레이션이 핵심 위험으로 남아 있다고 설명한다. 따라서 완전한 위험회피가 아니라, 위험자산을 유지하되 변동성 제어가 필요하다.
4. 국내 ETF 시장 자료는 2026년에도 AI와 반도체가 핵심이지만, AI 반도체·AI 전력 인프라·피지컬 AI·배당/금융을 조합하는 바벨 전략이 더 설득력 있다고 본다.
5. 한경머니 ETF 설문은 2026년 AI와 반도체가 핵심 키워드이고, 액티브 ETF 순자산이 전년 대비 66.9% 증가했으며, 변동성 큰 장세에서는 CD금리 ETF와 머니마켓 ETF 같은 현금성 자산을 보유하는 것이 필요하다는 의견을 소개한다.

참고 자료: Fidelity Business Cycle Investing, Funds Society/BofA Global Fund Manager Survey, Axios/BofA May 2026 FMS, Asia Business Daily 2026 ETF Barbell Strategy, 한경머니 2026 ETF 시장 트렌드, 삼성증권 Derivatives Weekly 2026.05.08.

### 이 포트폴리오에 적용하는 해석

현재 전략의 핵심은 **AI를 버리는 로테이션이 아니라, AI 코어를 유지하면서 과열 구간에서는 주변부를 순환·방어 자산으로 돌리는 로테이션**이다. 2026년 5월 22일 기준 KOSPI200 추세는 강하지만 VKOSPI가 66.97로 높기 때문에 위험자산을 70%까지 바로 채우지 않고 65%에서 시작한다.

| 역할 | ETF | 기본 비중 | 운용 판단 |
|---|---|---:|---|
| AI 코어 | KODEX AI반도체 | 4% | 국내 AI 반도체 직접성. 강한 모멘텀이지만 변동성이 높아 증액하지 않음 |
| AI 인프라 | KODEX AI전력핵심설비 | 3% | AI 데이터센터 전력 병목 수혜. 상위 종목 집중도가 높아 3% 제한 |
| 해외 AI 인프라 | KODEX 미국AI전력핵심인프라 | 4% | 국내 전력 테마와 지역 분산. 해외섹터 10% 상한 내에서 관리 |
| 순환주 | SOL 조선TOP3플러스 | 3% | 산업재 순환 후보. 최근 모멘텀은 약해 유지 관찰 |
| 정책/방산 | PLUS K방산 | 3% | 구조적 수출 테마. 최근 모멘텀 둔화로 증액은 보류 |
| 방어형 위험자산 | PLUS 고배당주, TIGER 미국배당다우존스 | 6% | 고변동성 구간에서 테마 포트폴리오의 완충 역할 |
| 비주식 헤지 | ACE KRX금현물 | 5% | 변동성 확대와 인플레이션 위험에 대한 완충 역할 |
| 현금성 방어 | CD금리, 머니마켓, 단기채 | 33% | 타깃볼 전략의 변동성 완충 장치 |
| 금리 하락 옵션 | ACE 미국30년국채액티브(H) | 2% | 핵심 안전자산이 아니라 위험회피/금리하락 옵션 |

### 로테이션 신호 규칙

월 1회 정기 리밸런싱을 원칙으로 하되, 월중에는 아래 신호가 2개 이상 동시에 발생할 때만 조정한다. 대회 회전율 조건을 충족하되 과매매로 보이지 않도록 월간 회전율은 10~20% 범위로 관리한다.

| 신호 | 확대 조건 | 축소 조건 |
|---|---|---|
| 시장 추세 | KOSPI200이 20일·60일 이동평균 위 | KOSPI200이 60일선 이탈 |
| 변동성 | VKOSPI 60 이하로 안정 | VKOSPI 70 초과 + 추세 훼손/NAV 손실 시 방어, VKOSPI 80 초과 + 추세 훼손 또는 NAV/MDD 방어 단계 시 강제 방어 |
| 테마 모멘텀 | ETF 1개월·3개월 수익률이 KOSPI200 또는 동일 테마 후보 대비 우위 | 1개월 수익률 음전환 또는 3개월 상대성과 급락 |
| 유동성 | 20일 평균 거래대금 유지 또는 증가 | 거래대금 급감, 괴리율 확대 |
| 리스크 | 구성종목 중복 낮고 상위 5개 집중도 감내 가능 | 동일 종목/동일 테마 중복 심화, 개별 ETF -12% 손실 |
| 펀더멘털/정책 | 실적 상향, 정책 수혜, CAPEX 확대 확인 | 실적 하향, 정책 모멘텀 약화 |

### 월별 운용안

| 시점 | 기본 행동 | 조건부 조정 |
|---|---|---|
| 6월 초 | 초기 포트폴리오 진입. 위험자산 65%, 안전자산 35% | AI반도체·AI전력은 유지하되 국내 섹터 상한 15%를 넘기지 않음 |
| 6월 말/7월 초 | 첫 로테이션 점검 | VKOSPI 60 이하이면 해외섹터 8%에서 10%까지 확대 가능. 국내 섹터는 기존 15% 상한 내에서 교체만 허용 |
| 7월 중 | 테마 과열/소외 확인 | AI반도체가 과열되고 원자력·2차전지·배당이 상대적으로 개선되면 국내 섹터 내부에서 1~3% 교체 |
| 8월 초 | 성과 방어 전환 검토 | 누적 수익이 크면 AI/전력/반도체 일부 이익실현 후 배당·금·CD/MMF 확대 |
| 8월 말 | 최종 마감 리스크 축소 | 순위 블라인드 전 과도한 고변동 테마를 줄이고 현금성 비중 확대 |

### 현재 기준 결론

1. 지금 당장 로테이션의 중심은 `KODEX AI반도체`, `KODEX AI전력핵심설비`, `KODEX 미국AI전력핵심인프라`이다.
2. `TIGER 미국필라델피아반도체나스닥`과 `KODEX 미국반도체`는 성과가 강하지만, 국내 AI반도체와 나스닥100에 이미 반도체/빅테크 베타가 있어 초기 포트폴리오에서는 제외한다.
3. `SOL 조선TOP3플러스`와 `PLUS K방산`은 순환/정책 테마로 유지하되, 6월 말까지 상대성과가 계속 약하면 원자력 또는 2차전지 후보와 교체를 검토한다.
4. `KODEX 은행`과 `TIGER 은행고배당플러스TOP10`은 가치/배당 로테이션 후보지만 현재 1개월·3개월 모멘텀이 약해 바로 편입하지 않는다.
5. 2차전지는 하반기 로테이션 후보로 관찰한다. `KODEX 2차전지산업`, `TIGER 2차전지테마`, `TIGER 2차전지소재Fn` 운용보고서와 구성종목은 확인 완료했으며, 실제 편입은 7월 상대모멘텀과 최신 운용보고서 확인 후 판단한다.



In [15]:
rotation_watch = pd.DataFrame([
    {'Bucket': 'AI 코어', 'Ticker': 'A395160', 'Action': '유지', 'Rule': '강한 모멘텀이나 변동성 높아 4% 유지'},
    {'Bucket': 'AI 전력', 'Ticker': 'A487240', 'Action': '유지', 'Rule': 'AI 전력 직접성 높지만 집중도 높아 3% 유지'},
    {'Bucket': '해외 AI 인프라', 'Ticker': 'A487230', 'Action': '유지/확대 후보', 'Rule': 'VKOSPI 60 이하 안정 시 해외섹터 상한 10%까지 확대 후보'},
    {'Bucket': '해외 반도체', 'Ticker': 'A381180', 'Action': '제외', 'Rule': 'SOX 대표 ETF이나 국내 AI반도체·나스닥100과 중복'},
    {'Bucket': '조선', 'Ticker': 'A466920', 'Action': '유지 관찰', 'Rule': '상대 모멘텀 둔화. 6월 말까지 회복 없으면 교체 검토'},
    {'Bucket': '방산', 'Ticker': 'A449450', 'Action': '유지 관찰', 'Rule': '정책/수출 테마 유지. 단기 모멘텀 약해 증액 보류'},
    {'Bucket': '원자력', 'Ticker': 'A434730', 'Action': '교체 후보', 'Rule': 'AI 전력 확장 후보. AI전력 ETF 과열 시 1~2% 교체 후보'},
    {'Bucket': '2차전지', 'Ticker': 'A305720', 'Action': '관찰 대표', 'Rule': '운용보고서/방법론 확인 완료. 7월 상대모멘텀 개선 시 국내 섹터 1~3% 교체 후보'},
    {'Bucket': '국내 배당', 'Ticker': 'A161510', 'Action': '유지/방어 확대 후보', 'Rule': 'VKOSPI 70 초과 + 추세 훼손/NAV 방어 전환 시 국내 섹터 내부 방어 확대 후보'},
    {'Bucket': '은행', 'Ticker': 'A091170', 'Action': '보류', 'Rule': '가치/배당 후보이나 현재 1M·3M 모멘텀 약함'},
    {'Bucket': '해외 배당', 'Ticker': 'A458730', 'Action': '유지', 'Rule': '해외 위험자산 내 방어형 역할'},
    {'Bucket': '금', 'Ticker': 'A411060', 'Action': '유지', 'Rule': '주식 테마와 중복 없는 비주식 헤지'},
])

rotation_watch = rotation_watch.merge(
    valid[['Ticker', 'Name', 'Class2', 'Theme', 'AUM', 'Expense Ratio', 'Ret_1M', 'Ret_3M', 'Ret_6M', 'Vol_20D', 'TradingValue_20D']],
    on='Ticker',
    how='left'
)
rotation_watch[['Bucket', 'Ticker', 'Name', 'Action', 'Class2', 'Ret_1M', 'Ret_3M', 'Ret_6M', 'Vol_20D', 'TradingValue_20D', 'Rule']]



,Bucket,Ticker,Name,Action,Class2,Ret_1M,Ret_3M,Ret_6M,Vol_20D,TradingValue_20D,Rule
0,AI 코어,A395160,KODEX AI반도체,유지,국내주식_섹터,0.378623,0.702111,1.634650,0.612449,2.259665e+11,강한 모멘텀이나 변동성 높아 4% 유지
1,AI 전력,A487240,KODEX AI전력핵심설비,유지,국내주식_섹터,0.426574,0.818872,1.257972,0.841278,6.190343e+11,AI 전력 직접성 높지만 집중도 높아 3% 유지
2,해외 AI 인프라,A487230,KODEX 미국AI전력핵심인프라,유지/확대 후보,해외주식_섹터,0.124793,0.365391,0.547952,0.482784,5.887268e+10,VKOSPI 60 이하 안정 시 해외섹터 상한 10%까지 확대 후보
3,해외 반도체,A381180,TIGER 미국필라델피아반도체나스닥,제외,해외주식_지수,0.283691,0.535026,0.846041,0.381216,9.908396e+10,SOX 대표 ETF이나 국내 AI반도체·나스닥100과 중복
4,조선,A466920,SOL 조선TOP3플러스,유지 관찰,국내주식_섹터,0.015174,0.033279,0.074971,0.434062,1.829848e+11,상대 모멘텀 둔화. 6월 말까지 회복 없으면 교체 검토
5,방산,A449450,PLUS K방산,유지 관찰,국내주식_섹터,-0.076227,0.080272,0.349449,0.407595,8.050382e+10,정책/수출 테마 유지. 단기 모멘텀 약해 증액 보류
6,원자력,A434730,HANARO 원자력iSelect,교체 후보,국내주식_섹터,0.061540,0.246091,0.670263,0.609198,8.949205e+10,AI 전력 확장 후보. AI전력 ETF 과열 시 1~2% 교체 후보
7,2차전지,A305720,KODEX 2차전지산업,관찰 대표,국내주식_섹터,-0.069520,0.100515,0.226315,0.556559,2.140647e+11,운용보고서/방법론 확인 완료. 7월 상대모멘텀 개선 시 국내 섹터 1~3% 교체 후보
8,국내 배당,A161510,PLUS 고배당주,유지/방어 확대 후보,국내주식_섹터,-0.008195,-0.040048,0.311278,0.265005,5.459424e+10,VKOSPI 70 초과 + 추세 훼손/NAV 방어 전환 시 국내 섹터 내부 방어 확...
9,은행,A091170,KODEX 은행,보류,국내주식_섹터,-0.038185,-0.087860,0.222939,0.238917,2.405355e+10,가치/배당 후보이나 현재 1M·3M 모멘텀 약함


## 15. 백테스트 및 벤치마크 수익률

현재 포트폴리오에 포함된 모든 ETF의 가격 데이터가 존재하는 공통 구간을 기준으로 백테스트를 수행한다. 2025년 5월 리포트와 2025.05.30까지 확인 가능한 가격·거래대금 정보만 사용해 초기 포트폴리오를 정하고, 그 이전 기간으로 성과를 소급하지 않는다.

이번 보강에서는 Target Vol Overlay의 효과를 단정하지 않고, **분산/섹터선정 효과**와 **Target Vol/운용규칙 효과**를 분리해 검증한다. 기준 벤치마크는 월별 목표비중 복원이 아니라 최초 65:35 매수 후 리밸런싱 없이 보유하는 방식이다.

| 구분 | 의미 |
|---|---|
| 계획서형 Target Vol Overlay 규칙 적용 | VKOSPI 고변동성+추세 훼손/NAV·MDD 손실 조건, 밴드 점검, 제한적 섹터 로테이션 적용 |
| 초기 65:35 포트폴리오 (리밸런싱 없음) | 2025년 5월 정보로 정한 초기 포트폴리오를 그대로 매수 후 보유 |
| TIGER 200 65% + CD금리 35% (리밸런싱 없음) | 국내 대형주와 현금성 자산만 사용한 단순 분산 벤치마크 |
| TIGER 200 100% | TIGER 200 ETF 단일 보유 벤치마크 |
| KOSPI200 100% | KOSPI200 지수 벤치마크 |

성과지표는 `Portfolio | TotalReturn | CAGR | Vol | Sharpe | Sortino | MDD | Calmar` 형식으로 출력한다. Sharpe와 Sortino는 동일한 Rf/MAR 기준을 적용하며, Rf는 상단 파라미터 셀에서 지정한 CD금리 ETF 평균 수익률 프록시를 사용한다.


In [16]:
import math

BACKTEST_OUTPUT_PATH = DATA_DIR / 'GAPS_백테스트_벤치마크_20260526.xlsx'

BACKTEST_INITIAL_PORTFOLIO = pd.DataFrame([
    # 2025년 5월 자료만 보고 정한 백테스트 전용 초기 포트폴리오.
    # 2026년 제출용 포트폴리오와 분리해 미래정보 편향을 줄인다.
    {'Ticker': 'A102110', 'Weight': 0.17, 'Role': '2025 백테스트 국내 코어: TIGER 200'},
    {'Ticker': 'A069500', 'Weight': 0.03, 'Role': '2025 백테스트 국내 코어 보조: KODEX 200'},
    {'Ticker': 'A379800', 'Weight': 0.11, 'Role': '2025 백테스트 해외 코어: 미국 S&P500'},
    {'Ticker': 'A133690', 'Weight': 0.06, 'Role': '2025 백테스트 해외 성장: NASDAQ100'},
    {'Ticker': 'A395160', 'Weight': 0.02, 'Role': '2025 백테스트 국내 AI반도체'},
    {'Ticker': 'A487240', 'Weight': 0.03, 'Role': '2025 백테스트 국내 AI전력'},
    {'Ticker': 'A466920', 'Weight': 0.03, 'Role': '2025 백테스트 조선'},
    {'Ticker': 'A449450', 'Weight': 0.03, 'Role': '2025 백테스트 방산'},
    {'Ticker': 'A434730', 'Weight': 0.02, 'Role': '2025 백테스트 원자력'},
    {'Ticker': 'A466940', 'Weight': 0.02, 'Role': '2025 백테스트 은행/배당'},
    {'Ticker': 'A487230', 'Weight': 0.05, 'Role': '2025 백테스트 해외 AI전력'},
    {'Ticker': 'A390390', 'Weight': 0.03, 'Role': '2025 백테스트 해외 반도체'},
    {'Ticker': 'A411060', 'Weight': 0.05, 'Role': '2025 백테스트 금 현물'},
    {'Ticker': 'A459580', 'Weight': 0.15, 'Role': '2025 백테스트 CD 금리'},
    {'Ticker': 'A488770', 'Weight': 0.10, 'Role': '2025 백테스트 머니마켓'},
    {'Ticker': 'A157450', 'Weight': 0.07, 'Role': '2025 백테스트 단기통안채'},
    {'Ticker': 'A423160', 'Weight': 0.03, 'Role': '2025 백테스트 KOFR 초단기 금리'},
])

portfolio_meta = BACKTEST_INITIAL_PORTFOLIO.merge(
    universe[['Ticker', 'Name', 'Class1', 'Class2']],
    on='Ticker',
    how='left'
)
strategy_tickers = portfolio_meta['Ticker'].tolist()

first_dates = price[strategy_tickers].apply(lambda s: s.dropna().index.min())
last_dates = price[strategy_tickers].apply(lambda s: s.dropna().index.max())
BACKTEST_START = max(BACKTEST_START_REQUESTED, first_dates.max())
BACKTEST_END = min(BACKTEST_END_REQUESTED, last_dates.min(), market_index.index.max(), vkospi.index.max())

print('Backtest period:', BACKTEST_START.date(), '~', BACKTEST_END.date())
print('Common-period driver:', first_dates.idxmax(), code_names.get(first_dates.idxmax()), first_dates.max().date())

risk_base = portfolio_meta.loc[portfolio_meta['Class1'] == '위험', ['Ticker', 'Weight']].set_index('Ticker')['Weight']
safe_base = portfolio_meta.loc[portfolio_meta['Class1'] == '안전', ['Ticker', 'Weight']].set_index('Ticker')['Weight']
risk_base_norm = risk_base / risk_base.sum()
safe_base_norm = safe_base / safe_base.sum()


def scaled_weights_by_risk_budget(risk_weights_norm, safe_weights_norm, risk_budget):
    risk_scaled = risk_weights_norm * risk_budget
    safe_scaled = safe_weights_norm * (1 - risk_budget)
    w = pd.concat([risk_scaled, safe_scaled]).sort_index()
    return w / w.sum()


def target_weights_by_risk_budget(risk_budget):
    return scaled_weights_by_risk_budget(risk_base_norm, safe_base_norm, risk_budget)


def month_first_dates(index):
    idx = pd.Index(index).sort_values()
    return idx.to_series().groupby(idx.to_period('M')).first().tolist()


def trading_cost_rates(tickers):
    # 대회 공식 NAV 기준: 국내 ETF 거래수수료 0.1%, 제세금 없음.
    # 슬리피지는 성과표 비용으로 차감하지 않고 주문 체결 리스크로 별도 관리한다.
    return pd.Series({ticker: COMMISSION_RATE + TAX_RATE for ticker in tickers}, dtype=float)


def estimate_rebalance_cost(current_value, target_weights, nav_before_cost, tickers):
    target_value = target_weights.reindex(tickers).fillna(0.0) * nav_before_cost
    trades_value = target_value - current_value.reindex(tickers).fillna(0.0)
    turnover_by_ticker = trades_value.abs() / nav_before_cost
    rates = trading_cost_rates(tickers)
    cost = float((turnover_by_ticker * rates).sum() * nav_before_cost)
    turnover = float(turnover_by_ticker.sum())
    cost_rate = float(cost / nav_before_cost) if nav_before_cost else np.nan
    return cost, turnover, cost_rate


def fixed_weight_nav(price_df, target_weights, start=BACKTEST_START, end=BACKTEST_END,
                     rebalance_monthly=True, apply_costs=False, portfolio_name='Portfolio'):
    tickers = list(target_weights.index)
    px = price_df.loc[start:end, tickers].ffill().dropna()
    target_weights = target_weights.reindex(tickers).astype(float)
    target_weights = target_weights / target_weights.sum()
    rebal_dates = set(month_first_dates(px.index)) if rebalance_monthly else set()
    holdings = None
    nav_rows = []
    cost_rows = []

    for dt, row in px.iterrows():
        if holdings is None:
            nav = 1.0
            if apply_costs and INCLUDE_INITIAL_TRADING_COST:
                zero_value = pd.Series(0.0, index=tickers)
                cost, turnover, cost_rate = estimate_rebalance_cost(zero_value, target_weights, nav, tickers)
                nav_after_cost = nav - cost
                cost_rows.append({
                    'Date': dt,
                    'Portfolio': portfolio_name,
                    'Turnover': turnover,
                    'TradingCost': cost,
                    'TradingCostRate': cost_rate,
                })
            else:
                nav_after_cost = nav
            holdings = target_weights * nav_after_cost / row
            nav = float((holdings * row).sum())
        else:
            nav = float((holdings * row).sum())
            if dt in rebal_dates:
                current_value = holdings * row
                cost, turnover, cost_rate = estimate_rebalance_cost(current_value, target_weights, nav, tickers)
                nav_after_cost = nav - cost if apply_costs else nav
                holdings = target_weights * nav_after_cost / row
                nav = float((holdings * row).sum())
                cost_rows.append({
                    'Date': dt,
                    'Portfolio': portfolio_name,
                    'Turnover': turnover,
                    'TradingCost': cost if apply_costs else 0.0,
                    'TradingCostRate': cost_rate if apply_costs else 0.0,
                })
        nav_rows.append((dt, nav))

    nav = pd.Series(dict(nav_rows), name=portfolio_name).sort_index()
    costs = pd.DataFrame(cost_rows)
    return nav, costs


def latest_signal_snapshot_for_trade(trade_dt):
    # 룩어헤드 방지: trade_dt 당일 종가를 쓰지 않고, trade_dt 이전에 확정된 마지막 종가 신호만 사용한다.
    vk_hist = vkospi.loc[vkospi.index < trade_dt, 'VKOSPI'].dropna()
    kospi_hist = market_index['코스피 200'].loc[market_index.index < trade_dt].dropna()

    if vk_hist.empty:
        signal_date = pd.NaT
        v = np.nan
        v20 = np.nan
        v60 = np.nan
    else:
        signal_date = vk_hist.index[-1]
        v = float(vk_hist.iloc[-1])
        v20 = float(vk_hist.tail(20).mean()) if len(vk_hist) >= 20 else np.nan
        v60 = float(vk_hist.tail(60).mean()) if len(vk_hist) >= 60 else np.nan

    if kospi_hist.empty:
        kospi = np.nan
        ma20 = np.nan
        ma60 = np.nan
        above20 = np.nan
        above60 = np.nan
    else:
        kospi = float(kospi_hist.iloc[-1])
        ma20 = float(kospi_hist.tail(20).mean()) if len(kospi_hist) >= 20 else np.nan
        ma60 = float(kospi_hist.tail(60).mean()) if len(kospi_hist) >= 60 else np.nan
        above20 = bool(kospi > ma20) if not pd.isna(ma20) else np.nan
        above60 = bool(kospi > ma60) if not pd.isna(ma60) else np.nan

    return {
        'Signal_Date': signal_date,
        'VKOSPI': v,
        'VKOSPI_20D_MA': v20,
        'VKOSPI_60D_MA': v60,
        'KOSPI200_Close': kospi,
        'KOSPI200_MA20': ma20,
        'KOSPI200_MA60': ma60,
        'KOSPI200_Above_MA20': above20,
        'KOSPI200_Above_MA60': above60,
    }


def risk_budget_from_trade_signal(trade_dt):
    snap = latest_signal_snapshot_for_trade(trade_dt)
    v = snap['VKOSPI']
    above20 = snap['KOSPI200_Above_MA20']
    above60 = snap['KOSPI200_Above_MA60']
    trend_damaged = above20 is False

    if pd.isna(v):
        risk_budget = PLAN_RULES_BASE_RISK_BUDGET
        signal = 'VKOSPI 없음: 기본 65%'
    elif v > 80 and trend_damaged:
        risk_budget = PLAN_RULES_FORCED_DEFENSE_RISK_BUDGET
        signal = 'VKOSPI 80 초과 + KOSPI200 추세 훼손: 강한 방어'
    elif v > 70 and trend_damaged:
        risk_budget = PLAN_RULES_DEFENSE_RISK_BUDGET
        signal = 'VKOSPI 70 초과 + KOSPI200 20일선 이탈: 방어 전환'
    elif v > 70:
        risk_budget = PLAN_RULES_BASE_RISK_BUDGET
        signal = 'VKOSPI 70 초과 단독: 고변동성 상승장, 위험자산 유지'
    elif v <= 60 and above20 is True and above60 is True:
        risk_budget = PLAN_RULES_EXPANSION_RISK_BUDGET
        signal = 'VKOSPI 60 이하 + KOSPI200 20/60일선 상회: 위험 확대'
    else:
        risk_budget = PLAN_RULES_BASE_RISK_BUDGET
        signal = '고변동성 또는 추세 확인 부족: 기본 65%'

    info = {**snap, 'Risk_Budget': risk_budget, 'Signal': signal}
    return risk_budget, info

def vkospi_risk_budget(trade_dt):
    risk_budget, info = risk_budget_from_trade_signal(trade_dt)
    return target_weights_by_risk_budget(risk_budget), info


def monthly_dynamic_nav(price_df, tickers, weight_func, start=BACKTEST_START, end=BACKTEST_END,
                        apply_costs=False, portfolio_name='Strategy_TargetVol_Rule'):
    px = price_df.loc[start:end, tickers].ffill().dropna()
    rebal_dates = set(month_first_dates(px.index))
    holdings = None
    nav_rows = []
    schedule_rows = []

    for dt, row in px.iterrows():
        if holdings is None:
            nav = 1.0
            current_weights, budget_info = weight_func(dt)
            current_weights = current_weights.reindex(tickers).fillna(0.0)
            if apply_costs and INCLUDE_INITIAL_TRADING_COST:
                zero_value = pd.Series(0.0, index=tickers)
                cost, turnover, cost_rate = estimate_rebalance_cost(zero_value, current_weights, nav, tickers)
                nav_after_cost = nav - cost
            else:
                cost, turnover, cost_rate = 0.0, 0.0, 0.0
                nav_after_cost = nav
            holdings = current_weights * nav_after_cost / row
            nav = float((holdings * row).sum())
            schedule_rows.append({'Date': dt, 'Rebalance_Type': 'Initial', 'Turnover': turnover, 'TradingCost': cost if apply_costs else 0.0, 'TradingCostRate': cost_rate if apply_costs else 0.0, **budget_info})
        else:
            nav = float((holdings * row).sum())
            if dt in rebal_dates:
                new_weights, budget_info = weight_func(dt)
                new_weights = new_weights.reindex(tickers).fillna(0.0)
                current_value = holdings * row
                cost, turnover, cost_rate = estimate_rebalance_cost(current_value, new_weights, nav, tickers)
                nav_after_cost = nav - cost if apply_costs else nav
                holdings = new_weights * nav_after_cost / row
                nav = float((holdings * row).sum())
                schedule_rows.append({
                    'Date': dt,
                    'Rebalance_Type': 'Monthly/Signal',
                    'Turnover': turnover,
                    'TradingCost': cost if apply_costs else 0.0,
                    'TradingCostRate': cost_rate if apply_costs else 0.0,
                    **budget_info,
                })
        nav_rows.append((dt, nav))

    nav = pd.Series(dict(nav_rows), name=portfolio_name).sort_index()
    schedule = pd.DataFrame(schedule_rows).drop_duplicates('Date').set_index('Date')
    return nav, schedule


def normalized_series(series, start=BACKTEST_START, end=BACKTEST_END, name=None):
    s = series.loc[start:end].dropna().astype(float)
    s = s / s.iloc[0]
    s.name = name or series.name
    return s


def risk_free_inputs(start=BACKTEST_START, end=BACKTEST_END):
    if RISK_FREE_RATE_ANNUAL is None:
        rf_price = price[RF_PROXY_TICKER].loc[start:end].ffill().dropna()
        rf_returns = rf_price.pct_change().dropna()
        rf_daily = float(rf_returns.mean())
        rf_ann = float((1 + rf_daily) ** TRADING_DAYS - 1)
        rf_source = f'{RF_PROXY_TICKER} {code_names.get(RF_PROXY_TICKER, RF_PROXY_DESCRIPTION)}: 백테스트 기간 일별 수익률 평균의 252영업일 연환산'
    else:
        rf_ann = float(RISK_FREE_RATE_ANNUAL)
        rf_daily = float((1 + rf_ann) ** (1 / TRADING_DAYS) - 1)
        rf_source = '상단 파라미터 RISK_FREE_RATE_ANNUAL 고정값'

    if MAR_ANNUAL is None:
        mar_ann = rf_ann
        mar_daily = rf_daily
        mar_source = 'Sortino MAR = Rf'
    else:
        mar_ann = float(MAR_ANNUAL)
        mar_daily = float((1 + mar_ann) ** (1 / TRADING_DAYS) - 1)
        mar_source = '상단 파라미터 MAR_ANNUAL 고정값'

    return rf_daily, rf_ann, mar_daily, mar_ann, rf_source, mar_source


rf_daily, rf_ann, mar_daily, mar_ann, rf_source, mar_source = risk_free_inputs()
print('Risk-free rate used:', f'{rf_ann:.2%}', '|', rf_source)
print('Sortino MAR used:', f'{mar_ann:.2%}', '|', mar_source)


# ===== 계획서 규칙형 Target Vol / 로테이션 백테스트 =====
# 기존 monthly_dynamic_nav는 단순 위험예산 시뮬레이션이다.
# 아래 plan_rules_nav는 계획서의 월간 점검, NAV/MDD 방어, 섹터 로테이션 후보,
# 밴드 리밸런싱을 수치화한 별도 실행 엔진이다.
ROTATION_CANDIDATE_TICKERS = [
    'A305720',  # KODEX 2차전지산업
    'A434730',  # HANARO 원자력iSelect
    'A445290',  # KODEX 로봇액티브
    'A091170',  # KODEX 은행
    'A453640',  # KODEX 미국S&P500헬스케어
    'A453630',  # KODEX 미국S&P500필수소비재
]
PLAN_TICKERS = sorted(set(strategy_tickers + [t for t in ROTATION_CANDIDATE_TICKERS if t in price.columns]))

plan_meta = universe[['Ticker', 'Name', 'Class1', 'Class2']].drop_duplicates('Ticker').set_index('Ticker')
PLAN_RISK_TICKERS = [t for t in PLAN_TICKERS if plan_meta.loc[t, 'Class1'] == '위험']
PLAN_SAFE_TICKERS = [t for t in PLAN_TICKERS if plan_meta.loc[t, 'Class1'] == '안전']
PLAN_DOMESTIC_SECTOR_TICKERS = [t for t in PLAN_TICKERS if plan_meta.loc[t, 'Class2'] == '국내주식_섹터']

DEFENSE_REDUCTION_ORDER = [
    'A466920',  # SOL 조선TOP3플러스
    'A449450',  # PLUS K방산
    'A487240',  # KODEX AI전력핵심설비
    'A229200',  # KODEX 코스닥150
    'A395160',  # KODEX AI반도체
    'A133690',  # TIGER 미국나스닥100
    'A487230',  # KODEX 미국AI전력핵심인프라
    'A458730',  # TIGER 미국배당다우존스
    'A411060',  # ACE KRX금현물
    'A102110',  # TIGER 200
    'A379800',  # KODEX 미국S&P500
]
CASH_RECEIVER_WEIGHTS = pd.Series({
    'A459580': 0.40,  # KODEX CD금리액티브
    'A488770': 0.30,  # KODEX 머니마켓액티브
    'A157450': 0.15,  # TIGER 단기통안채
    'A0061Z0': 0.10,  # RISE 단기특수은행채액티브
    'A453850': 0.05,  # ACE 미국30년국채액티브(H)
})
RISK_EXPANSION_WEIGHTS = pd.Series({'A102110': 0.50, 'A379800': 0.30, 'A487230': 0.20})


def normalize_weights(weights):
    w = weights.reindex(PLAN_TICKERS).fillna(0.0).clip(lower=0.0)
    total = float(w.sum())
    return w / total if total else w


def risk_weight_of(weights):
    return float(weights.reindex(PLAN_RISK_TICKERS).fillna(0.0).sum())


def safe_weight_of(weights):
    return float(weights.reindex(PLAN_SAFE_TICKERS).fillna(0.0).sum())


def add_to_cash(weights, amount):
    w = weights.copy()
    receivers = CASH_RECEIVER_WEIGHTS.reindex(w.index).dropna()
    receivers = receivers / receivers.sum()
    for ticker, share in receivers.items():
        w.loc[ticker] = w.get(ticker, 0.0) + amount * share
    return normalize_weights(w)


def fund_from_safe(weights, amount):
    w = weights.copy()
    remaining = amount
    for ticker in ['A459580', 'A488770', 'A157450', 'A0061Z0', 'A453850']:
        if ticker not in w.index or remaining <= 0:
            continue
        sell = min(float(w.loc[ticker]), remaining)
        w.loc[ticker] -= sell
        remaining -= sell
    return w, amount - remaining


def reduce_risk_to_budget(weights, target_risk, reason):
    w = normalize_weights(weights)
    current_risk = risk_weight_of(w)
    cut_needed = max(0.0, current_risk - target_risk)
    if cut_needed <= 1e-9:
        return w, []

    remaining = cut_needed
    actions = []
    for ticker in DEFENSE_REDUCTION_ORDER:
        if ticker not in w.index or remaining <= 1e-9:
            continue
        sell = min(float(w.loc[ticker]), remaining)
        if sell > 1e-9:
            w.loc[ticker] -= sell
            remaining -= sell
            actions.append(f'{reason}: {ticker} {sell:.2%} 축소')
    executed = cut_needed - remaining
    w = add_to_cash(w, executed)
    return normalize_weights(w), actions


def increase_risk_to_budget(weights, target_risk, reason):
    w = normalize_weights(weights)
    current_risk = risk_weight_of(w)
    add_needed = max(0.0, target_risk - current_risk)
    if add_needed <= 1e-9:
        return w, []
    w, funded = fund_from_safe(w, add_needed)
    if funded <= 1e-9:
        return normalize_weights(w), []

    add_targets = RISK_EXPANSION_WEIGHTS.reindex(w.index).dropna()
    add_targets = add_targets / add_targets.sum()
    for ticker, share in add_targets.items():
        w.loc[ticker] = w.get(ticker, 0.0) + funded * share
    return normalize_weights(w), [f'{reason}: 위험자산 {funded:.2%} 확대']


def trailing_return_on(ticker, signal_date, days):
    if ticker not in price.columns or pd.isna(signal_date):
        return np.nan
    s = price[ticker].loc[:signal_date].dropna()
    if len(s) <= days:
        return np.nan
    return float(s.iloc[-1] / s.iloc[-days - 1] - 1)


def kospi200_trailing_return(signal_date, days):
    s = market_index['코스피 200'].loc[:signal_date].dropna()
    if len(s) <= days:
        return np.nan
    return float(s.iloc[-1] / s.iloc[-days - 1] - 1)


def exchange_weight(weights, from_ticker, to_ticker, amount, reason):
    w = weights.copy()
    if from_ticker not in w.index or to_ticker not in w.index:
        return w, []
    trade = min(float(w.loc[from_ticker]), amount)
    if trade <= 1e-9:
        return w, []
    w.loc[from_ticker] -= trade
    w.loc[to_ticker] = w.get(to_ticker, 0.0) + trade
    return normalize_weights(w), [f'{reason}: {from_ticker}->{to_ticker} {trade:.2%}']


def apply_sector_rotation(weights, trade_dt, signal_info, nav_state):
    # 월간 점검에서만 작동한다. 방어 상태/VKOSPI 70 초과에서는 신규 2차전지·로봇 편입을 막는다.
    w = normalize_weights(weights)
    actions = []
    signal_date = signal_info.get('Signal_Date')
    if pd.isna(signal_date) or trade_dt < pd.Timestamp('2025-07-01'):
        return w, actions
    if signal_info.get('VKOSPI', np.nan) > 70 or nav_state in ['방어 전환', '강제 방어']:
        return w, ['방어 상태: 신규 테마 로테이션 보류']

    kospi_1m = kospi200_trailing_return(signal_date, 21)
    kospi_3m = kospi200_trailing_return(signal_date, 63)

    # 조선·방산 1개월/3개월 상대열위 + 2차전지 개선 시 1~3%p 교체.
    for donor in ['A466920', 'A449450']:
        donor_1m = trailing_return_on(donor, signal_date, 21)
        donor_3m = trailing_return_on(donor, signal_date, 63)
        battery_1m = trailing_return_on('A305720', signal_date, 21)
        battery_3m = trailing_return_on('A305720', signal_date, 63)
        if all(pd.notna(x) for x in [donor_1m, donor_3m, battery_1m, battery_3m, kospi_1m, kospi_3m]):
            donor_weak = donor_1m < kospi_1m and donor_3m < kospi_3m
            battery_improving = battery_1m > 0 and battery_3m > 0 and battery_1m > donor_1m and battery_3m > donor_3m
            if donor_weak and battery_improving:
                w, a = exchange_weight(w, donor, 'A305720', 0.015, '국내 섹터 상대모멘텀 로테이션')
                actions.extend(a)

    # AI전력 과열 후 원자력 상대모멘텀 개선 시 1%p 제한 교체.
    ai_power_3m = trailing_return_on('A487240', signal_date, 63)
    nuclear_1m = trailing_return_on('A434730', signal_date, 21)
    ai_power_1m = trailing_return_on('A487240', signal_date, 21)
    if all(pd.notna(x) for x in [ai_power_3m, nuclear_1m, ai_power_1m]):
        if ai_power_3m > 0.30 and nuclear_1m > ai_power_1m:
            w, a = exchange_weight(w, 'A487240', 'A434730', 0.01, 'AI전력 과열 완화 로테이션')
            actions.extend(a)

    return normalize_weights(w), actions


def apply_limit_bands(weights, current_weights, signal_info):
    # 평가비중 초과는 즉시 전량 복원하지 않는다.
    # 월간 점검에서는 soft band를, 월중에는 hard band 또는 추세 훼손 동반 초과를 중심으로 일부 이익실현한다.
    w = normalize_weights(weights.copy())
    actions = []
    trend_damaged = signal_info.get('KOSPI200_Above_MA20') is False

    risk_w = risk_weight_of(current_weights)
    if risk_w > PLAN_RULES_RISK_SOFT_BAND:
        w, a = reduce_risk_to_budget(
            current_weights,
            PLAN_RULES_EXPANSION_RISK_BUDGET,
            f'위험자산 {PLAN_RULES_RISK_SOFT_BAND:.0%} 초과 완화 밴드'
        )
        actions.extend(a)

    domestic_sector_w = float(current_weights.reindex(PLAN_DOMESTIC_SECTOR_TICKERS).fillna(0.0).sum())
    if domestic_sector_w > PLAN_RULES_DOMESTIC_SECTOR_SOFT_BAND:
        sector_weights = current_weights.reindex(PLAN_DOMESTIC_SECTOR_TICKERS).fillna(0.0)
        top_ticker = sector_weights.idxmax()
        cut = min(float(w.get(top_ticker, 0.0)), domestic_sector_w - 0.16)
        if cut > 1e-9:
            w.loc[top_ticker] -= cut
            w = add_to_cash(w, cut)
            suffix = ' + 추세 훼손' if trend_damaged else ''
            actions.append(f'국내주식_섹터 {PLAN_RULES_DOMESTIC_SECTOR_SOFT_BAND:.0%} 초과 밴드{suffix}: {top_ticker} {cut:.2%} 이익실현')

    for ticker, weight in current_weights.items():
        if weight > PLAN_RULES_SINGLE_ETF_SOFT_BAND and ticker in w.index:
            cut = min(float(w.loc[ticker]), weight - PLAN_RULES_SINGLE_ETF_TRIM_TARGET)
            if cut > 1e-9:
                w.loc[ticker] -= cut
                w = add_to_cash(w, cut)
                suffix = ' + 추세 훼손' if trend_damaged else ''
                actions.append(f'개별 ETF {PLAN_RULES_SINGLE_ETF_SOFT_BAND:.0%} 초과 밴드{suffix}: {ticker} {cut:.2%} 이익실현')

    return normalize_weights(w), actions

def monthly_band_rebalance(current_weights, desired_weights, band=PLAN_RULES_REBALANCE_BAND_ABS):
    # 목표와 1%p 이상 벌어진 종목만 조정하고, 잔여 차이는 CD/MMF가 흡수한다.
    current = normalize_weights(current_weights)
    desired = normalize_weights(desired_weights)
    new_w = current.copy()
    for ticker in desired.index:
        if abs(float(current.get(ticker, 0.0) - desired.get(ticker, 0.0))) >= band:
            new_w.loc[ticker] = desired.loc[ticker]
    diff = 1.0 - float(new_w.sum())
    if diff > 0:
        new_w = add_to_cash(new_w, diff)
    elif diff < 0:
        new_w, _ = fund_from_safe(new_w, -diff)
    return normalize_weights(new_w)


def nav_defense_state(nav, peak_nav):
    nav_loss = nav - 1.0
    mdd = nav / peak_nav - 1.0 if peak_nav else 0.0
    if nav_loss <= -0.07 or mdd <= -0.08:
        return '강제 방어', PLAN_RULES_FORCED_DEFENSE_RISK_BUDGET, nav_loss, mdd
    if nav_loss <= -0.05 or mdd <= -0.06:
        return '방어 전환', PLAN_RULES_DEFENSE_RISK_BUDGET, nav_loss, mdd
    if nav_loss <= -0.03 or mdd <= -0.04:
        return '주의', PLAN_RULES_BASE_RISK_BUDGET, nav_loss, mdd
    return '정상', PLAN_RULES_BASE_RISK_BUDGET, nav_loss, mdd


def expansion_allowed(signal_info, nav_state):
    if not PLAN_RULES_ALLOW_RISK_EXPANSION or nav_state != '정상':
        return False
    if signal_info.get('VKOSPI', np.nan) > 60:
        return False
    if signal_info.get('KOSPI200_Above_MA20') is not True or signal_info.get('KOSPI200_Above_MA60') is not True:
        return False
    if PLAN_RULES_REQUIRE_FALLING_VKOSPI_FOR_EXPANSION:
        v = signal_info.get('VKOSPI', np.nan)
        v20 = signal_info.get('VKOSPI_20D_MA', np.nan)
        v60 = signal_info.get('VKOSPI_60D_MA', np.nan)
        return pd.notna(v) and pd.notna(v20) and pd.notna(v60) and v < v20 < v60
    return True


def desired_plan_weights(current_weights, trade_dt, signal_info, nav_state, nav_target_risk, monthly_check):
    w = normalize_weights(current_weights)
    actions = []

    v = signal_info.get('VKOSPI', np.nan)
    trend_damaged = signal_info.get('KOSPI200_Above_MA20') is False
    nav_pressure = nav_state in ['주의', '방어 전환', '강제 방어']
    signal_risk_target = PLAN_RULES_BASE_RISK_BUDGET

    if pd.notna(v) and v > 80 and (trend_damaged or nav_pressure):
        signal_risk_target = PLAN_RULES_FORCED_DEFENSE_RISK_BUDGET
        actions.append('VKOSPI 80 초과 + 추세 훼손/NAV 손실: 강제 방어')
    elif pd.notna(v) and v > 70 and (trend_damaged or nav_pressure):
        signal_risk_target = PLAN_RULES_DEFENSE_RISK_BUDGET
        actions.append('VKOSPI 70 초과 + 추세 훼손/NAV 손실: 방어 전환')
    elif trend_damaged and nav_pressure:
        signal_risk_target = PLAN_RULES_DEFENSE_RISK_BUDGET
        actions.append('NAV 주의 + KOSPI200 20일선 이탈: 방어 전환')
    elif pd.notna(v) and v > 70:
        actions.append('VKOSPI 70 초과 단독: 위험자산 유지, 신규 고베타 증액 보류')
    elif monthly_check and expansion_allowed(signal_info, nav_state):
        signal_risk_target = PLAN_RULES_EXPANSION_RISK_BUDGET
        actions.append('월간 점검: VKOSPI 하락 안정 + 추세 유지, 위험자산 단계적 확대')

    target_risk = signal_risk_target if nav_state == '정상' else min(signal_risk_target, nav_target_risk)
    if risk_weight_of(w) > target_risk:
        w, a = reduce_risk_to_budget(w, target_risk, f'{nav_state}/Target Vol 위험예산 {target_risk:.0%}')
        actions.extend(a)
    elif monthly_check and risk_weight_of(w) < target_risk:
        w, a = increase_risk_to_budget(w, target_risk, f'월간 점검 Target Vol 위험예산 {target_risk:.0%}')
        actions.extend(a)

    if monthly_check:
        w, a = apply_sector_rotation(w, trade_dt, signal_info, nav_state)
        actions.extend(a)

    return normalize_weights(w), actions, target_risk

def plan_rules_nav(price_df, start=BACKTEST_START, end=BACKTEST_END, apply_costs=False,
                   portfolio_name='Strategy_TargetVol_Rule'):
    px = price_df.loc[start:end, PLAN_TICKERS].ffill().dropna()
    rebal_dates = set(month_first_dates(px.index))
    weights0 = BACKTEST_INITIAL_PORTFOLIO.set_index('Ticker')['Weight'].reindex(PLAN_TICKERS).fillna(0.0)
    weights0 = normalize_weights(weights0)

    holdings = None
    decision_holdings = None
    nav_rows = []
    schedule_rows = []
    peak_nav = 1.0

    for dt, row in px.iterrows():
        if holdings is None:
            nav = 1.0
            decision_nav = 1.0
            if apply_costs and INCLUDE_INITIAL_TRADING_COST:
                zero_value = pd.Series(0.0, index=PLAN_TICKERS)
                cost, turnover, cost_rate = estimate_rebalance_cost(zero_value, weights0, nav, PLAN_TICKERS)
                nav_after_cost = nav - cost
            else:
                cost, turnover, cost_rate = 0.0, 0.0, 0.0
                nav_after_cost = nav
            holdings = weights0 * nav_after_cost / row
            decision_holdings = weights0 * decision_nav / row
            nav = float((holdings * row).sum())
            decision_nav = float((decision_holdings * row).sum())
            signal_info = latest_signal_snapshot_for_trade(dt)
            schedule_rows.append({
                'Date': dt,
                'Rebalance_Type': 'Initial',
                'NAV_State': '정상',
                'NAV_Loss': decision_nav - 1.0,
                'MDD_ToDate': min(decision_nav / peak_nav - 1.0, 0.0),
                'Risk_Budget': risk_weight_of(weights0),
                'Portfolio_Risk_Weight': risk_weight_of(weights0),
                'Signal': '초기 65:35 진입',
                'Actions': '초기 65:35 진입',
                'Turnover': turnover,
                'TradingCost': cost if apply_costs else 0.0,
                'TradingCostRate': cost_rate if apply_costs else 0.0,
                **signal_info,
            })
            nav_rows.append((dt, nav))
            continue

        nav = float((holdings * row).sum())
        decision_nav = float((decision_holdings * row).sum())
        peak_nav = max(peak_nav, decision_nav)
        current_value = holdings * row
        decision_value = decision_holdings * row
        current_weights = normalize_weights(decision_value / decision_nav)
        signal_info = latest_signal_snapshot_for_trade(dt)
        nav_state, nav_target_risk, nav_loss, mdd = nav_defense_state(decision_nav, peak_nav)

        monthly_check = dt in rebal_dates
        v = signal_info.get('VKOSPI', np.nan)
        trend_damaged = signal_info.get('KOSPI200_Above_MA20') is False
        vkospi_defense_event = pd.notna(v) and v > 70 and (trend_damaged or nav_state != '정상')
        mid_event = vkospi_defense_event or nav_state in ['방어 전환', '강제 방어']
        risk_w = risk_weight_of(current_weights)
        domestic_sector_w = float(current_weights.reindex(PLAN_DOMESTIC_SECTOR_TICKERS).fillna(0.0).sum())
        single_hard_event = any(
            (weight > PLAN_RULES_SINGLE_ETF_HARD_BAND) or
            (weight > PLAN_RULES_SINGLE_ETF_SOFT_BAND and trend_damaged)
            for weight in current_weights
        )
        band_event = (
            risk_w > PLAN_RULES_RISK_HARD_BAND or
            (risk_w > PLAN_RULES_RISK_SOFT_BAND and trend_damaged) or
            domestic_sector_w > PLAN_RULES_DOMESTIC_SECTOR_HARD_BAND or
            (domestic_sector_w > PLAN_RULES_DOMESTIC_SECTOR_SOFT_BAND and trend_damaged) or
            single_hard_event
        )

        if monthly_check or mid_event or band_event:
            desired, actions, target_risk = desired_plan_weights(
                current_weights, dt, signal_info, nav_state, nav_target_risk, monthly_check
            )
            desired, band_actions = apply_limit_bands(desired, current_weights, signal_info)
            actions.extend(band_actions)
            if monthly_check:
                desired = monthly_band_rebalance(current_weights, desired)

            cost, turnover, cost_rate = estimate_rebalance_cost(current_value, desired, nav, PLAN_TICKERS)
            nav_after_cost = nav - cost if apply_costs else nav
            holdings = desired * nav_after_cost / row
            decision_holdings = desired * decision_nav / row
            nav = float((holdings * row).sum())
            current_weights = desired
            schedule_rows.append({
                'Date': dt,
                'Rebalance_Type': 'Monthly' if monthly_check else ('MidRisk' if mid_event else 'Band'),
                'NAV_State': nav_state,
                'NAV_Loss': nav_loss,
                'MDD_ToDate': mdd,
                'Risk_Budget': target_risk,
                'Portfolio_Risk_Weight': risk_weight_of(current_weights),
                'Signal': '; '.join(actions) if actions else '밴드 내 유지',
                'Actions': '; '.join(actions) if actions else '밴드 내 유지',
                'Turnover': turnover,
                'TradingCost': cost if apply_costs else 0.0,
                'TradingCostRate': cost_rate if apply_costs else 0.0,
                **signal_info,
            })

        nav_rows.append((dt, nav))

    nav = pd.Series(dict(nav_rows), name=portfolio_name).sort_index()
    schedule = pd.DataFrame(schedule_rows).drop_duplicates('Date').set_index('Date')
    return nav, schedule

static_weights = BACKTEST_INITIAL_PORTFOLIO.set_index('Ticker')['Weight']

# 계획서 규칙형: 월중 조정은 VKOSPI 고변동성+추세 훼손/NAV/MDD 또는 완화 밴드의 hard 조건에서만 발생한다.
strategy_tv_gross, targetvol_schedule_gross = plan_rules_nav(
    price, apply_costs=False, portfolio_name='Strategy_TargetVol_Rule'
)
strategy_tv_net, targetvol_schedule = plan_rules_nav(
    price, apply_costs=True, portfolio_name='Strategy_TargetVol_Rule'
)

bench_65_35_gross, costs_bench_gross = fixed_weight_nav(
    price, pd.Series({'A102110': 0.65, 'A459580': 0.35}), rebalance_monthly=False, apply_costs=False, portfolio_name='Benchmark_65_35_TIGER200_CD'
)
bench_65_35_net, costs_bench_net = fixed_weight_nav(
    price, pd.Series({'A102110': 0.65, 'A459580': 0.35}), rebalance_monthly=False, apply_costs=True, portfolio_name='Benchmark_65_35_TIGER200_CD'
)

static_no_rebal_gross, costs_no_rebal_gross = fixed_weight_nav(
    price, static_weights, rebalance_monthly=False, apply_costs=False, portfolio_name='Static_65_35_NoRebalance'
)
static_no_rebal_net, costs_no_rebal_net = fixed_weight_nav(
    price, static_weights, rebalance_monthly=False, apply_costs=True, portfolio_name='Static_65_35_NoRebalance'
)

tiger200_100_gross = normalized_series(price['A102110'], name='TIGER200_100')
tiger200_100_net, costs_tiger200_100_net = fixed_weight_nav(
    price, pd.Series({'A102110': 1.0}), rebalance_monthly=False, apply_costs=True, portfolio_name='TIGER200_100'
)
kospi200_100 = normalized_series(market_index['코스피 200'], name='KOSPI200_100')

nav_gross = pd.concat([
    strategy_tv_gross,
    bench_65_35_gross,
    tiger200_100_gross,
    kospi200_100,
    static_no_rebal_gross,
], axis=1).dropna()

# KOSPI200은 지수 벤치마크이므로 비용을 적용하지 않는다. TIGER 200 단일 ETF는 최초 매수 수수료를 반영한다.
nav_net = pd.concat([
    strategy_tv_net,
    bench_65_35_net,
    tiger200_100_net,
    kospi200_100,
    static_no_rebal_net,
], axis=1).dropna()

cost_events = pd.concat([
    targetvol_schedule.reset_index()[['Date', 'Turnover', 'TradingCost', 'TradingCostRate']].assign(Portfolio='Strategy_TargetVol_Rule'),
    costs_bench_net,
    costs_tiger200_100_net,
    costs_no_rebal_net,
], ignore_index=True, sort=False)

PORTFOLIO_LABELS = {
    'Strategy_TargetVol_Rule': '계획서형 Target Vol Overlay 규칙 적용',
    'Static_65_35_NoRebalance': '초기 65:35 포트폴리오 (리밸런싱 없음)',
    'Benchmark_65_35_TIGER200_CD': 'TIGER 200 65% + CD금리 35% (리밸런싱 없음)',
    'TIGER200_100': 'TIGER 200 100%',
    'KOSPI200_100': 'KOSPI200 100%',
}
REQUESTED_COLUMNS = ['Portfolio', 'TotalReturn', 'CAGR', 'Vol', 'Sharpe', 'Sortino', 'MDD', 'Calmar']


def max_drawdown(nav_series, initial_base=1.0):
    # 최초 매수 수수료가 첫 NAV에 반영되면 시작 원금 1.0 대비 낙폭도 MDD에 포함한다.
    running_peak = nav_series.cummax().clip(lower=initial_base)
    dd = nav_series / running_peak - 1
    return float(dd.min())


def period_return(nav_series, periods):
    s = nav_series.dropna()
    if len(s) <= periods:
        return np.nan
    return float(s.iloc[-1] / s.iloc[-periods - 1] - 1)


def ytd_return(nav_series):
    s = nav_series.dropna()
    y = s.loc[s.index >= pd.Timestamp('2026-01-01')]
    if len(y) < 2:
        return np.nan
    return float(y.iloc[-1] / y.iloc[0] - 1)


def downside_deviation(daily_returns, mar_daily):
    downside = np.minimum(daily_returns - mar_daily, 0.0)
    if len(downside) == 0:
        return np.nan
    return float(np.sqrt(np.mean(np.square(downside))) * np.sqrt(TRADING_DAYS))


def perf_stats(nav_df, labels=None, scenario=None, rf_ann_used=rf_ann, mar_ann_used=mar_ann, mar_daily_used=mar_daily):
    labels = labels or PORTFOLIO_LABELS
    rows = []
    for col in nav_df.columns:
        s = nav_df[col].dropna()
        r = s.pct_change().dropna()
        total = float(s.iloc[-1] / 1.0 - 1)
        cagr = float((1 + total) ** (TRADING_DAYS / len(r)) - 1) if len(r) else np.nan
        ann_vol = float(r.std() * np.sqrt(TRADING_DAYS)) if len(r) else np.nan
        sharpe = float((cagr - rf_ann_used) / ann_vol) if ann_vol and ann_vol > 0 else np.nan
        dd = max_drawdown(s)
        calmar = float(cagr / abs(dd)) if dd < 0 else np.nan
        down_dev = downside_deviation(r, mar_daily_used)
        sortino = float((cagr - mar_ann_used) / down_dev) if down_dev and down_dev > 0 else np.nan
        rows.append({
            'Scenario': scenario,
            'Portfolio_ID': col,
            'Portfolio': labels.get(col, col),
            'Start': s.index[0].date(),
            'End': s.index[-1].date(),
            'Obs': len(s),
            'TotalReturn': total,
            'CAGR': cagr,
            'Vol': ann_vol,
            'Sharpe': sharpe,
            'Sortino': sortino,
            'MDD': dd,
            'Calmar': calmar,
            'Ret_1M': period_return(s, 21),
            'Ret_3M': period_return(s, 63),
            'Ret_6M': period_return(s, 126),
            'YTD_2026': ytd_return(s),
            'Final_NAV': float(s.iloc[-1]),
        })
    return pd.DataFrame(rows)


def format_metric_table(df):
    out = df[REQUESTED_COLUMNS].copy()
    for c in ['TotalReturn', 'CAGR', 'Vol', 'MDD']:
        out[c] = out[c].map(lambda x: '' if pd.isna(x) else f'{x:.2%}')
    for c in ['Sharpe', 'Sortino', 'Calmar']:
        out[c] = out[c].map(lambda x: '' if pd.isna(x) else f'{x:.2f}')
    return out


performance_gross = perf_stats(nav_gross, scenario='비용 반영 전')
performance_net = perf_stats(nav_net, scenario='비용 반영 후')

portfolio_order = list(PORTFOLIO_LABELS.values())
result_table_gross = performance_gross.set_index('Portfolio').loc[portfolio_order].reset_index()[REQUESTED_COLUMNS]
result_table_net = performance_net.set_index('Portfolio').loc[portfolio_order].reset_index()[REQUESTED_COLUMNS]
result_table_gross_fmt = format_metric_table(result_table_gross)
result_table_net_fmt = format_metric_table(result_table_net)

# 기존 저장 로직과 호환되도록 대표 성과표 변수도 유지한다.
performance_summary = performance_net.copy()
trailing_returns = performance_gross[['Portfolio', 'Ret_1M', 'Ret_3M', 'Ret_6M', 'YTD_2026', 'TotalReturn']].copy()
nav = nav_net.copy()


def build_effect_decomposition(perf_df, scenario):
    p = perf_df.set_index('Portfolio')
    pairs = [
        ('분산/섹터선정 효과', 'TIGER 200 65% + CD금리 35% (리밸런싱 없음)', '초기 65:35 포트폴리오 (리밸런싱 없음)'),
        ('Target Vol/운용규칙 효과', '초기 65:35 포트폴리오 (리밸런싱 없음)', '계획서형 Target Vol Overlay 규칙 적용'),
    ]
    rows = []
    for effect, from_name, to_name in pairs:
        row = {'Scenario': scenario, 'Effect': effect, 'From': from_name, 'To': to_name}
        for metric in ['Vol', 'MDD', 'Sharpe', 'Sortino']:
            row[f'{metric}_Diff'] = p.loc[to_name, metric] - p.loc[from_name, metric]
        rows.append(row)
    return pd.DataFrame(rows)


effect_decomposition = pd.concat([
    build_effect_decomposition(performance_gross, '비용 반영 전'),
    build_effect_decomposition(performance_net, '비용 반영 후'),
], ignore_index=True)

def format_effect_table(df):
    out = df.copy()
    for c in ['Vol_Diff', 'MDD_Diff']:
        out[c] = out[c].map(lambda x: '' if pd.isna(x) else f'{x:+.2%}')
    for c in ['Sharpe_Diff', 'Sortino_Diff']:
        out[c] = out[c].map(lambda x: '' if pd.isna(x) else f'{x:+.2f}')
    return out


effect_decomposition_fmt = format_effect_table(effect_decomposition)

lookahead_audit = targetvol_schedule.reset_index()[[
    'Date', 'Signal_Date', 'VKOSPI', 'VKOSPI_20D_MA', 'VKOSPI_60D_MA',
    'KOSPI200_Close', 'KOSPI200_MA20', 'KOSPI200_MA60',
    'KOSPI200_Above_MA20', 'KOSPI200_Above_MA60', 'Risk_Budget', 'Signal',
    'Turnover', 'TradingCostRate'
]].copy()
lookahead_audit['Execution_Assumption'] = SIGNAL_EXECUTION_ASSUMPTION

backtest_footnotes = pd.DataFrame([
    {'Item': '백테스트 기간', 'Note': f'{BACKTEST_START.date()} ~ {BACKTEST_END.date()} ({len(nav_gross)}거래일). 2025년 5월 말 정보로 구성한 포트폴리오를 다음 영업일부터 적용한 약 12개월 상승장 구간.'},
    {'Item': 'Rf/Sharpe', 'Note': f'{rf_source}. 적용 Rf={rf_ann:.2%}.'},
    {'Item': 'Sortino MAR', 'Note': f'{mar_source}. 적용 MAR={mar_ann:.2%}, 하방편차는 일별 수익률이 일별 MAR을 밑돈 부분만 사용.'},
    {'Item': '리밸런싱 가격', 'Note': f'{REBALANCE_EXECUTION_ASSUMPTION}. {PRICE_EXECUTION_PROXY}.'},
    {'Item': 'VKOSPI 신호', 'Note': '체결일 Date에는 Date 직전 Signal_Date의 종가 신호만 사용한다. 20일/60일 이동평균도 Signal_Date까지의 trailing 데이터로 계산한다.'},
    {'Item': '계획서 규칙형 Target Vol', 'Note': '월중 조정은 VKOSPI 70 초과가 KOSPI200 추세 훼손 또는 NAV/MDD 손실과 결합될 때, NAV -5%/MDD -6% 방어 전환, NAV -7%/MDD -8% 강제 방어, 또는 완화 밴드 hard 조건에서만 수행. 섹터 로테이션은 월간 점검일에만 실행.'},
    {'Item': '위험확대 70% 조건', 'Note': 'Target Vol 효과 분리표에서는 65% 초과 위험확대를 OFF로 두어 방어 overlay만 검증한다. 별도 민감도 분석에서 켤 경우 VKOSPI 60 이하, KOSPI200 20/60일선 상회, VKOSPI < 20일 평균 < 60일 평균일 때만 허용.'},
    {'Item': '비교 벤치마크', 'Note': '초기 65:35 포트폴리오와 TIGER 200 65%+CD 35% 벤치마크는 월별 목표비중 복원이 아니라 최초 비중 매수 후 리밸런싱 없이 보유하는 방식으로 계산.'},
    {'Item': '거래비용', 'Note': f'대회 기준 국내 ETF 거래수수료 {COMMISSION_RATE:.2%}, 제세금 {TAX_RATE:.2%}. 슬리피지는 공식 NAV 비용으로 차감하지 않음. 최초 진입 및 이후 회전 금액에 비례해 차감.'},
    {'Item': '비용 전/후 비교', 'Note': '동일 신호·동일 거래 경로에서 비용만 분리해 비교한다. KOSPI200 지수 벤치마크는 비용 미적용, TIGER 200 ETF 벤치마크는 최초 매수 수수료 반영.'},
    {'Item': '회전율 공식', 'Note': '회전율(%) = 총 매매금액 / 평균자산 × 100. 대회 시작일부터 5영업일 이내 80% 이상, 월 회전율 10% 이상, 3개월 누적 기준 없음.'},
    {'Item': 'KOSPI200 100%', 'Note': '지수 벤치마크이므로 거래비용을 적용하지 않음. TIGER 200 100% ETF 벤치마크에는 최초 매수 수수료를 반영.'},
])

data_coverage = pd.DataFrame({
    'Ticker': first_dates.index,
    'Name': [code_names.get(t) for t in first_dates.index],
    'First_Date': first_dates.values,
    'Last_Date': last_dates.values,
}).sort_values('First_Date', ascending=False)

period_limitations = pd.DataFrame([
    {'Check': '주 백테스트 구간', 'Result': f'{BACKTEST_START.date()} ~ {BACKTEST_END.date()}. 2025년 5월 말까지의 정보로 정한 백테스트 포트폴리오를 다음 영업일부터 적용.'},
    {'Check': '원 포트폴리오 장기 검증 제약', 'Result': f'백테스트 전용 초기 포트폴리오의 공통 시작일은 {first_dates.max().date()}이며, 시작 요청일은 2025년 5월 말 정보가 반영되는 {BACKTEST_START_REQUESTED.date()}로 설정.'},
    {'Check': '다른 국면 검증 한계', 'Result': '2022 하락장, 2020 변동성 급등 구간은 현재 편입 ETF 일부가 상장 전이라 원 포트폴리오 그대로는 재현 불가. 대체 ETF 프록시 장기 백테스트는 제출본의 주 성과표에서 제외.'},
])





Backtest period: 2025-06-02 ~ 2026-05-22
Common-period driver: A488770 KODEX 머니마켓액티브 2024-08-06
Risk-free rate used: 2.80% | A459580 KODEX CD금리액티브(합성): 백테스트 기간 일별 수익률 평균의 252영업일 연환산
Sortino MAR used: 2.80% | Sortino MAR = Rf


In [17]:
# 비용 반영 전/후 성과표, 효과분해, 룩어헤드 점검, footnote
print('비용 반영 전 성과표')
display(result_table_gross_fmt)

print('비용 반영 후 성과표')
display(result_table_net_fmt)

print('두 효과 분리 검증: 분산 효과와 Target Vol 효과')
display(effect_decomposition_fmt)

print('룩어헤드 방지 점검: Date 체결에는 직전 Signal_Date 종가 신호만 사용')
display(lookahead_audit)

print('계획서 규칙형 Target Vol 실행 로그')
display(targetvol_schedule.reset_index()[[
    'Date', 'Rebalance_Type', 'NAV_State', 'Risk_Budget', 'Portfolio_Risk_Weight',
    'NAV_Loss', 'MDD_ToDate', 'VKOSPI', 'Actions', 'Turnover', 'TradingCostRate'
]])

print('결과표 footnote')
display(backtest_footnotes)

print('기간 제약 및 데이터 커버리지')
display(period_limitations)
display(data_coverage.head(8))


비용 반영 전 성과표


,Portfolio,TotalReturn,CAGR,Vol,Sharpe,Sortino,MDD,Calmar
0,계획서형 Target Vol Overlay 규칙 적용,77.50%,84.06%,16.82%,4.83,7.29,-8.37%,10.04
1,초기 65:35 포트폴리오 (리밸런싱 없음),98.11%,106.87%,22.37%,4.65,7.15,-10.71%,9.98
2,TIGER 200 65% + CD금리 35% (리밸런싱 없음),161.10%,177.45%,32.09%,5.44,8.53,-17.06%,10.40
3,TIGER 200 100%,246.42%,274.76%,40.23%,6.76,10.64,-20.63%,13.32
4,KOSPI200 100%,240.86%,268.37%,40.57%,6.55,10.31,-21.13%,12.70


비용 반영 후 성과표


,Portfolio,TotalReturn,CAGR,Vol,Sharpe,Sortino,MDD,Calmar
0,계획서형 Target Vol Overlay 규칙 적용,77.12%,83.64%,16.83%,4.80,7.24,-8.40%,9.96
1,초기 65:35 포트폴리오 (리밸런싱 없음),97.91%,106.65%,22.37%,4.64,7.13,-10.71%,9.96
2,TIGER 200 65% + CD금리 35% (리밸런싱 없음),160.83%,177.15%,32.09%,5.43,8.52,-17.06%,10.39
3,TIGER 200 100%,246.07%,274.36%,40.23%,6.75,10.63,-20.63%,13.30
4,KOSPI200 100%,240.86%,268.37%,40.57%,6.55,10.31,-21.13%,12.70


두 효과 분리 검증: 분산 효과와 Target Vol 효과


,Scenario,Effect,From,To,Vol_Diff,MDD_Diff,Sharpe_Diff,Sortino_Diff
0,비용 반영 전,분산/섹터선정 효과,TIGER 200 65% + CD금리 35% (리밸런싱 없음),초기 65:35 포트폴리오 (리밸런싱 없음),-9.73%,+6.35%,-0.79,-1.38
1,비용 반영 전,Target Vol/운용규칙 효과,초기 65:35 포트폴리오 (리밸런싱 없음),계획서형 Target Vol Overlay 규칙 적용,-5.55%,+2.34%,+0.18,+0.15
2,비용 반영 후,분산/섹터선정 효과,TIGER 200 65% + CD금리 35% (리밸런싱 없음),초기 65:35 포트폴리오 (리밸런싱 없음),-9.73%,+6.35%,-0.79,-1.38
3,비용 반영 후,Target Vol/운용규칙 효과,초기 65:35 포트폴리오 (리밸런싱 없음),계획서형 Target Vol Overlay 규칙 적용,-5.54%,+2.31%,+0.16,+0.11


... (5 outputs left out)

,Item,Note
0,백테스트 기간,2025-06-02 ~ 2026-05-22 (238거래일). 2025년 5월 말 정...
1,Rf/Sharpe,A459580 KODEX CD금리액티브(합성): 백테스트 기간 일별 수익률 평균의 ...
2,Sortino MAR,"Sortino MAR = Rf. 적용 MAR=2.80%, 하방편차는 일별 수익률이 ..."
3,리밸런싱 가격,"월말 종가로 신호 확정, 다음 영업일(월초) 시가에 리밸런싱. 현재 파일에는 일별 ..."
4,VKOSPI 신호,체결일 Date에는 Date 직전 Signal_Date의 종가 신호만 사용한다. 2...
5,계획서 규칙형 Target Vol,월중 조정은 VKOSPI 70 초과가 KOSPI200 추세 훼손 또는 NAV/MDD...
6,위험확대 70% 조건,Target Vol 효과 분리표에서는 65% 초과 위험확대를 OFF로 두어 방어 o...
7,비교 벤치마크,초기 65:35 포트폴리오와 TIGER 200 65%+CD 35% 벤치마크는 월별 ...
8,거래비용,"대회 기준 국내 ETF 거래수수료 0.10%, 제세금 0.00%. 슬리피지는 공식 ..."
9,비용 전/후 비교,동일 신호·동일 거래 경로에서 비용만 분리해 비교한다. KOSPI200 지수 벤치마...


기간 제약 및 데이터 커버리지


,Check,Result
0,주 백테스트 구간,2025-06-02 ~ 2026-05-22. 2025년 5월 말까지의 정보로 정한 ...
1,원 포트폴리오 장기 검증 제약,"백테스트 전용 초기 포트폴리오의 공통 시작일은 2024-08-06이며, 시작 요청일..."
2,다른 국면 검증 한계,"2022 하락장, 2020 변동성 급등 구간은 현재 편입 ETF 일부가 상장 전이라..."


,Ticker,Name,First_Date,Last_Date
14,A488770,KODEX 머니마켓액티브,2024-08-06,2026-05-22
10,A487230,KODEX 미국AI전력핵심인프라,2024-07-09,2026-05-22
5,A487240,KODEX AI전력핵심설비,2024-07-09,2026-05-22
9,A466940,TIGER 은행고배당플러스TOP10,2023-10-17,2026-05-22
6,A466920,SOL 조선TOP3플러스,2023-10-05,2026-05-22
13,A459580,KODEX CD금리액티브(합성),2023-06-08,2026-05-22
7,A449450,PLUS K방산,2023-01-05,2026-05-22
8,A434730,HANARO 원자력iSelect,2022-06-28,2026-05-22


## 15-1. 기간 제약

주 백테스트 구간은 `2025-06-02~2026-05-22`다. 2025년 5월 말까지 확인 가능한 자료로 백테스트 초기 포트폴리오를 정하고, 다음 영업일부터 운용을 시작한 것으로 가정한다. 구간은 약 12개월이지만 2025년 하반기~2026년 상반기 한국 주식 강세가 포함된 상승장 성격이 강하므로 결과를 장기 평균 성과로 해석하지 않는다.

2026년 제출용 원 포트폴리오는 일부 신규 상장 ETF가 포함되어 2022년 하락장이나 2020년 변동성 급등 구간을 같은 종목으로 재현할 수 없다. 대체 ETF 프록시로 기간을 늘린 장기 백테스트는 원 전략의 종목 선정 성과와 운용 규칙 효과가 섞일 수 있어 제출본의 주 성과표에서 제외한다.


In [18]:
# 백테스트 결과 저장
with pd.ExcelWriter(BACKTEST_OUTPUT_PATH, engine='openpyxl') as writer:
    performance_gross.to_excel(writer, sheet_name='Perf_Gross', index=False)
    performance_net.to_excel(writer, sheet_name='Perf_Net', index=False)
    result_table_gross.to_excel(writer, sheet_name='Table_Gross', index=False)
    result_table_net.to_excel(writer, sheet_name='Table_Net', index=False)
    effect_decomposition.to_excel(writer, sheet_name='Effects', index=False)
    trailing_returns.to_excel(writer, sheet_name='Trailing_Returns', index=False)
    nav_gross.to_excel(writer, sheet_name='Daily_NAV_Gross')
    nav_net.to_excel(writer, sheet_name='Daily_NAV_Net')
    targetvol_schedule.to_excel(writer, sheet_name='TargetVol_Schedule')
    targetvol_schedule.to_excel(writer, sheet_name='PlanRule_Schedule')
    lookahead_audit.to_excel(writer, sheet_name='Lookahead_Audit', index=False)
    backtest_footnotes.to_excel(writer, sheet_name='Footnotes', index=False)
    period_limitations.to_excel(writer, sheet_name='Period_Limits', index=False)
    cost_events.to_excel(writer, sheet_name='Cost_Events', index=False)
    portfolio_meta.to_excel(writer, sheet_name='Initial_Portfolio', index=False)
    data_coverage.to_excel(writer, sheet_name='Data_Coverage', index=False)

print(f'저장 완료: {BACKTEST_OUTPUT_PATH}')


저장 완료: input/GAPS_백테스트_벤치마크_20260526.xlsx
